# Buy the Rumor, Sell the News — ทำซ้ำงานวิจัยแบบย่อ (1 สัปดาห์)

ทำตาม **"แผนทำซ้ำงานวิจัย Buy the Rumor, Sell the News (1 สัปดาห์)"** ของงาน *Buy the Rumor, Sell the News: When Is News Priced In?* (arXiv:2608.14014) ครบ 6 ขั้นตอน
notebook นี้ **self-contained** (โค้ดทั้งหมดอยู่ในนี้) — แต่ละขั้นอ่าน parquet ของขั้นก่อนหน้าแล้วเขียนไฟล์ของตัวเองใน `data/rumor/btr_<MODE>/` จึงรันซ้ำเฉพาะขั้นที่แก้ได้

| ขั้น | ทำอะไร | ผลผลิต |
|---|---|---|
| 1 | เลือกหุ้น S&P 500 (3 กลุ่มขนาด × 45 ตัว กระจายอุตสาหกรรม), ดึง/กรอง FNSPID, ตรวจเวลา, ทำความสะอาด, ดึงราคา yfinance | `universe`, `news_clean`, `prices` |
| 2 | LLM ติดป้าย + คัดกรองในการเรียกครั้งเดียว (8 ป้าย + ขยะ 2, 3 คุณลักษณะ) | `news_tagged` |
| 3 | embedding → story (NEW / follow-up) → event (หุ้น, วันซื้อขาย, ป้าย) | `news_stories`, `events` |
| 4 | beta 252 วัน, AR, CAR 4 ช่วง, กลุ่มขนาด, bootstrap ตามวัน | `event_returns`, ตาราง raw, กราฟ |
| 5 | ค่าฐาน placebo (ข่าวเป็นกลาง / วันเงียบ) → adjusted, ข่าวดีเทียบข่าวร้าย | `baseline`, `event_returns_adj`, ตารางหลัก |
| 6 | NEW only, วันข่าวเดียว, รายปี, BH, ป้าย 2 โมเดล, benchmark 3 ปัจจัย, คุณลักษณะ | ตารางความทนทาน, ตาราง drift |

**โหมด** (`MODE` ใน cell config)
- `"test"` — ข่าว 1 เดือน (ม.ค. 2023) + LLM จำกัดแถว ใช้ตรวจว่า pipeline ทำงาน **ไม่ใช่ผลวิจัย**
- `"full"` — ข่าว 2021–2023 ตามแผน (ราคา ม.ค. 2020 – ก.พ. 2024), bootstrap 5,000 รอบ

ต้องมีใน `Research/.env`: `OPENROUTER_API_KEY` (และ `OPENROUTER_MODEL` ถ้าไม่ใช้ค่า default) — key เดียวกับ `LLMExtract.ipynb`

In [ ]:
import hashlib
import html
import io
import json
import os
import re
import threading
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from types import SimpleNamespace
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display
from pydantic import BaseModel, ConfigDict, Field, ValidationError

RESEARCH = Path.cwd().parent if Path.cwd().name == "ipynb" else Path.cwd()
load_dotenv(RESEARCH / ".env")

DATA = Path(os.environ.get("BTR_DATA") or RESEARCH / "data")   # ตั้ง BTR_DATA ใน .env ถ้าข้อมูลอยู่ที่อื่น
CACHE = DATA / "rumor" / "_cache"          # ใช้ร่วมกันทุกโหมด: ผล LLM, รายชื่อหุ้น, FNSPID ที่กรองแล้ว
CACHE.mkdir(parents=True, exist_ok=True)
# FNSPID: ข่าวที่ผูก ticker + เนื้อหาเต็มช่วง 2021-2023 อยู่ใน nasdaq_exteral_data.csv (23 GB, เรียงตาม ticker)
# (All_external.csv มีข่าวที่ tag หุ้นแค่ถึงกลางปี 2020 -> ใช้ไม่ได้กับแผนนี้)
RAW_FNSPID = DATA / "raw" / "fnspid" / "nasdaq_exteral_data.csv"
FNSPID_URL = "https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/Stock_news/nasdaq_exteral_data.csv"
FNSPID_SIZE = 23_232_979_597
SP500_URL = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv"

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_columns", 40)

## Config

In [ ]:
MODE = "full"            # "test" | "full"

CFG = dict(
    # ---- ช่วงเวลา (แผน: ข่าว 2021-2023, ราคา ต้นปี 2020 - ก.พ. 2024)
    news_start="2021-01-01", news_end="2023-12-31",
    price_start="2020-01-01", price_end="2024-03-01",
    universe_ref=("2020-01-01", "2020-12-31"),   # ปีที่ใช้จัดกลุ่มขนาดตอนเลือกหุ้น (ก่อนช่วงข่าว -> ไม่ look-ahead)
    # ---- ขั้นตอนที่ 1
    n_per_bucket=45,           # กลุ่มละ 40-50 ตัว x 3 กลุ่มขนาด
    universe_seed=42,
    min_articles=30,           # หุ้นต้องมีข่าว FNSPID อย่างน้อยเท่านี้ในช่วง 2021-2023 (กันเลือกหุ้นที่ไม่มีข่าวเลย)
    max_per_ticker_day=20,     # สุ่มเก็บไม่เกินนี้ต่อหุ้นต่อวัน
    article_chars=4000,        # เก็บเนื้อข่าวไม่เกินนี้ใน cache (LLM ใช้แค่ content_chars)
    # ---- ขั้นตอนที่ 2 (OpenRouter)
    model=os.environ.get("OPENROUTER_MODEL", "google/gemini-3.5-flash-lite"),
    second_model="anthropic/claude-haiku-5.5",   # โมเดลที่ 2 สำหรับตรวจป้าย (ขั้นตอนที่ 6)
    prompt_version="rumor-v4",
    content_chars=2500,
    llm_workers=8,
    llm_limit=None,            # None = ทุกแถว
    title_filter=True,         # กรองข่าวขยะจากหัวข่าวก่อนส่ง LLM (ส่วน 2.0)
    screen=True,               # รอบคัดกรอง (ส่วน 2.2b): prompt สั้นถามแค่ keep ก่อนส่งรอบเต็ม
    screen_model="google/gemini-3.5-flash-lite",   # โมเดลรอบคัดกรอง (ทดลอง 2.5 flash lite แล้ว: ผ่าน 71% -> แพงกว่า)
    screen_version="screen-v3",  # batch 10 (ตรวจแล้ว: ทิ้งข่าวที่ v4 เก็บ 4.5%, ผ่าน 31%) · screen-v2 = ทีละข่าว 6.3% / 22%
    screen_batch=10,           # ข่าวต่อการเรียก 1 ครั้งในรอบคัดกรอง (screen-v2 = 1, screen-v3 = 10)
    screen_chars=500,          # เนื้อข่าวต่อข่าวในรอบคัดกรอง (screen-v2 = 800, screen-v3 = 500 · รอบเต็มยังใช้ content_chars)
    screen_validate_v4=True,   # 2.2b: คัดกรองแถวที่มีป้าย v4 แล้ว (~1,800 แถว ~$0.3) เพื่อวัดว่าตัดข่าวที่ควรเก็บกี่ข่าว
    prompt_cache=True,         # cache system prompt ของรอบเต็ม (ข้อความเดิมทุกตัวอักษร, input ส่วนนั้นถูกลง ~90%)
    llm_budget_usd=40.0,       # ประมาณค่าใช้จ่ายเกินนี้ -> หยุดก่อนรันชุดเต็ม
    # ---- ขั้นตอนที่ 3
    embed_model="sentence-transformers/all-MiniLM-L6-v2",
    sim_threshold=0.80,        # ต้องปรับใหม่จาก calibration_pairs.csv
    # ---- ขั้นตอนที่ 4-5
    beta_window=252, beta_min_periods=126,
    winsor=0.01,               # ใช้ดูผลแบบ winsorize ประกอบ
    n_boot=5000,               # ทดลอง 1,000 / ผลสุดท้าย 5,000
    min_neutral_per_bucket=1000,   # ข่าวเป็นกลางน้อยกว่านี้ -> ใช้วันเงียบเป็นค่าฐาน
    n_quiet=50000,
    seed=0,
    # ---- ขั้นตอนที่ 6
    second_model_n=2000,
    factors3=("SPY", "GLD", "IEF"),
)
if MODE == "test":
    CFG.update(news_start="2023-01-01", news_end="2023-01-31", llm_limit=400, n_boot=500,
               min_neutral_per_bucket=30, n_quiet=5000, second_model_n=100)

C = SimpleNamespace(**CFG)
OUT = DATA / "rumor" / f"btr_{MODE}"
(OUT / "tables").mkdir(parents=True, exist_ok=True)
P = lambda name: OUT / name          # noqa: E731
print(MODE, "->", OUT)
print({k: v for k, v in CFG.items() if k not in ("model",)}, "| model:", C.model)

## ค่าคงที่และ helper ที่ใช้ทุกขั้น
- ป้าย 8 ประเภท + ขยะ 2 (แผนขั้นตอนที่ 2 ข้อ 3) · อายุ story แบบตายตัว (แผนขั้นตอนที่ 3 ข้อ 5)
- ช่วง CAR: pre [−5, 0], d0 [0], post1 [+1, +5], post2 [+6, +20]
- `cluster_bootstrap`: สุ่ม **วันซื้อขายทั้งวัน** แบบใส่คืน (หุ้นที่มีข่าววันเดียวกันขยับไปด้วยกัน)

In [ ]:
TAGS = {
    "earnings_guidance": "ผลประกอบการ + guidance",
    "analyst": "นักวิเคราะห์",
    "dividend_buyback": "ปันผล/ซื้อหุ้นคืน",
    "mna": "M&A",
    "legal_regulatory": "คดีความ/กฎระเบียบ",
    "product_partnership": "เปิดตัวสินค้า + พันธมิตร",
    "management": "ผู้บริหาร",
    "macro_other": "มหภาค/อื่น ๆ",
    "price_commentary": "วิจารณ์ราคา (ขยะ)",
    "promotional": "โฆษณา (ขยะ)",
}
TAG_IDS = list(TAGS)
JUNK_TAGS = {"price_commentary", "promotional"}
STORY_MAX_AGE = {"mna": 75, "legal_regulatory": 75, "management": 15, "earnings_guidance": 5, "analyst": 5}
STORY_DEFAULT_AGE = 2
BENCHMARKS = ["SPY", "GLD", "IEF", "IWM"]
WINDOWS = {"pre": (-5, 0), "d0": (0, 0), "post1": (1, 5), "post2": (6, 20)}
OFFSETS = list(range(-5, 21))
AR_COLS = [f"ar_{'m' if k < 0 else 'p'}{abs(k)}" for k in OFFSETS]      # ar_m5 ... ar_p20
CAR_COLS = {w: f"car_{w}" for w in WINDOWS}
ADJ_COLS = {w: f"adj_car_{w}" for w in WINDOWS}
NY = "America/New_York"
MIN_BOOT_DAYS = 5


def log(*a):
    print(*a, flush=True)


def short_hash(s: str, n: int = 12) -> str:
    return hashlib.sha1(str(s).encode("utf-8")).hexdigest()[:n]


def norm_title(s) -> str:
    return re.sub(r"[^a-z0-9]+", " ", html.unescape(str(s or "")).lower()).strip()


def trading_days(start="2015-01-01", end="2027-12-31") -> pd.DatetimeIndex:
    import pandas_market_calendars as mcal
    sched = mcal.get_calendar("NYSE").schedule(start_date=start, end_date=end)
    return pd.DatetimeIndex(sched.index).tz_localize(None).normalize()


CAL = trading_days()


def to_trade_date(ts_utc: pd.Series, date_only: pd.Series, cal=CAL) -> pd.Series:
    # กฎ 16:00 นิวยอร์ก: ข่าวหลังปิดตลาด / วันหยุด -> วันซื้อขายถัดไป
    # date_only (ไม่มีเวลา) -> ใช้วันที่ตามที่มี แล้วเลื่อนไปวันซื้อขายถัดไปถ้าเป็นวันหยุด
    ny = ts_utc.dt.tz_convert(NY)
    d = ts_utc.dt.tz_localize(None).dt.normalize().where(date_only, ny.dt.tz_localize(None).dt.normalize())
    after_close = (~date_only) & (ny.dt.hour >= 16)
    d = d + pd.to_timedelta(after_close.astype(int), unit="D")
    pos = cal.searchsorted(d.values)
    out = pd.Series(pd.NaT, index=ts_utc.index, dtype="datetime64[ns]")
    ok = pos < len(cal)
    out[ok] = cal[pos[ok]]
    return out


def cluster_bootstrap(values, clusters, n_boot=1000, seed=0) -> dict:
    # ค่าเฉลี่ย + p-value สองทาง จาก bootstrap ที่สุ่ม 'วันซื้อขาย' ทั้งวันแบบใส่คืน
    v = np.asarray(values, dtype=float)
    m = ~np.isnan(v)
    v, c = v[m], np.asarray(clusters)[m]
    out = {"n": int(len(v)), "n_days": 0, "mean": np.nan, "se": np.nan, "p": np.nan}
    if len(v) == 0:
        return out
    out["mean"] = float(v.mean())
    codes, uniq = pd.factorize(c)
    k = len(uniq)
    out["n_days"] = int(k)
    if k < MIN_BOOT_DAYS:
        return out
    sums = np.bincount(codes, weights=v, minlength=k)
    cnts = np.bincount(codes, minlength=k).astype(float)
    rng = np.random.default_rng(seed)
    bm = np.empty(n_boot)
    step = max(1, 2_000_000 // k)
    for s in range(0, n_boot, step):
        e = min(n_boot, s + step)
        idx = rng.integers(0, k, size=(e - s, k))
        bm[s:e] = sums[idx].sum(1) / cnts[idx].sum(1)
    out["se"] = float(bm.std(ddof=1))
    out["p"] = float((np.sum(np.abs(bm - bm.mean()) >= abs(out["mean"])) + 1) / (n_boot + 1))
    return out


def summarize(df, cols: dict, by=None, n_boot=None, seed=None, cluster="trade_date") -> pd.DataFrame:
    # ตาราง ค่าเฉลี่ย (%) + p-value แยกตามกลุ่ม x ช่วงเวลา. cols = {window: column}
    n_boot = n_boot or C.n_boot
    seed = C.seed if seed is None else seed
    groups = [("ALL", df)] if by is None else list(df.groupby(by, observed=True))
    rows = []
    for g, d in groups:
        for w, col in cols.items():
            r = cluster_bootstrap(d[col], d[cluster], n_boot, seed)
            rows.append({"group": "|".join(map(str, g)) if isinstance(g, tuple) else str(g), "window": w,
                         "n": r["n"], "n_days": r["n_days"], "mean_pct": r["mean"] * 100,
                         "se_pct": r["se"] * 100, "p": r["p"]})
    return pd.DataFrame(rows)


def wide(table, value="mean_pct"):
    t = table.pivot(index="group", columns="window", values=value)
    return t[[w for w in WINDOWS if w in t.columns]]


def bh_qvalues(p) -> np.ndarray:
    from statsmodels.stats.multitest import multipletests
    p = np.asarray(p, dtype=float)
    q = np.full_like(p, np.nan)
    m = ~np.isnan(p)
    if m.any():
        q[m] = multipletests(p[m], method="fdr_bh")[1]
    return q


def winsorize_cols(df, cols, q):
    df = df.copy()
    for c in cols:
        lo, hi = df[c].quantile([q, 1 - q])
        df[c] = df[c].clip(lo, hi)
    return df

---
# ขั้นตอนที่ 1: เตรียมข้อมูลข่าวและราคา (วันที่ 1)

1. **อ่าน FNSPID ทีละ chunk** เก็บเฉพาะหุ้น S&P 500 ปี 2021–2023 → cache parquet (ทำครั้งเดียว)
   ไฟล์ `nasdaq_exteral_data.csv` **เรียงตาม ticker** ถ้ายังดาวน์โหลดไม่ครบ จะอ่านเท่าที่มีแล้ว **ตัด ticker ตัวสุดท้าย (อาจถูกตัดกลาง) และทุกตัวหลังจากนั้น** ออกจาก universe
2. **ตรวจเวลาเผยแพร่** และนับข่าวต่อปี/ต่อหุ้น
3. **เลือกหุ้น**: เรียง S&P 500 ตามมูลค่าซื้อขายเฉลี่ยปี 2020 → 3 กลุ่ม → กลุ่มละ 45 ตัว round-robin ข้ามอุตสาหกรรม (ไม่เลือกตามจำนวนข่าว)
4. **ทำความสะอาด**: `article_id`, ลบซ้ำด้วย URL / หัวข่าว normalize, ticker ที่เปลี่ยนชื่อ, จำกัด 20 บทความ/หุ้น/วัน, `trade_date`
5. **ราคา**: yfinance ราคาปิดปรับแล้ว + volume ของหุ้นที่เลือก + SPY/GLD/IEF/IWM

In [ ]:
# ticker เก่า -> ticker ปัจจุบัน (ราคาใน yfinance อยู่ใต้ ticker ปัจจุบัน)
RENAMES = {"FB": "META", "ANTM": "ELV", "FISV": "FI", "RE": "EG", "PKI": "RVTY", "FLT": "CPAY", "ABC": "COR",
           "WLTW": "WTW", "DISCA": "WBD", "DISCK": "WBD", "BLL": "BALL", "NLOK": "GEN", "FBHS": "FBIN",
           "COG": "CTRA", "HFC": "DINO", "TWTR": None, "BRK-B": "BRK.B", "BF-B": "BF.B", "BRKB": "BRK.B"}
FNSPID_YEARS = ("2021", "2022", "2023")


def load_sp500() -> pd.DataFrame:
    path = CACHE / "sp500_constituents.csv"
    if not path.exists():
        r = requests.get(SP500_URL, timeout=60)
        r.raise_for_status()
        path.write_bytes(r.content)
    sp = pd.read_csv(path)
    return sp.rename(columns={"Symbol": "ticker", "Security": "name", "GICS Sector": "sector"})[["ticker", "name", "sector"]]


def download_prices(tickers, start, end, retries=3) -> pd.DataFrame:
    # yfinance ราคาปิดปรับแล้ว + volume -> long (date, ticker, close, volume); ตัวที่หลุดลองใหม่
    import yfinance as yf

    def _dl(tks):
        ymap = {t.replace(".", "-"): t for t in tks}
        ylist, out = list(ymap), []
        for i in range(0, len(ylist), 100):
            chunk = ylist[i:i + 100]
            data = yf.download(chunk, start=start, end=end, auto_adjust=True, progress=False, threads=True,
                               group_by="column")
            if data.empty:
                continue
            close, vol = data["Close"], data["Volume"]
            if isinstance(close, pd.Series):
                close, vol = close.to_frame(chunk[0]), vol.to_frame(chunk[0])
            long = close.stack().rename("close").to_frame().join(vol.stack().rename("volume")).reset_index()
            long.columns = ["date", "ticker", "close", "volume"]
            out.append(long)
        if not out:
            return pd.DataFrame(columns=["date", "ticker", "close", "volume"])
        px = pd.concat(out, ignore_index=True)
        px["ticker"] = px["ticker"].map(ymap)
        px["date"] = pd.to_datetime(px["date"]).dt.tz_localize(None).dt.normalize()
        return px.dropna(subset=["close"])

    tickers = list(tickers)
    px = _dl(tickers)
    for _ in range(retries):
        missing = sorted(set(tickers) - set(px["ticker"]))
        if not missing:
            break
        time.sleep(3)
        px = pd.concat([px, _dl(missing)], ignore_index=True)
    return px.drop_duplicates(["ticker", "date"]).sort_values(["ticker", "date"]).reset_index(drop=True)


sp500 = load_sp500()
print(len(sp500), "หุ้นใน S&P 500 (รายชื่อปัจจุบัน)")

### 1.1 อ่าน FNSPID ทีละ chunk → cache
ครั้งแรกใช้เวลานาน (อ่านไฟล์หลายสิบ GB) ครั้งต่อไปอ่านจาก `_cache/fnspid_sp500_2021_2023.parquet`

In [ ]:
FNSPID_CACHE = CACHE / "fnspid_sp500_2021_2023.parquet"
FNSPID_META = CACHE / "fnspid_sp500_2021_2023.json"


class _Limited(io.RawIOBase):
    # file object ที่อ่านได้ไม่เกิน limit ไบต์
    def __init__(self, f, limit):
        self.f, self.left = f, limit

    def readable(self):
        return True

    def readinto(self, b):
        if self.left <= 0:
            return 0
        n = self.f.readinto(memoryview(b)[:min(len(b), self.left)])
        self.left -= n
        return n


def _last_record_start(path, tail=64 << 20) -> int:
    # ตำแหน่งเริ่มของ record สุดท้าย (บรรทัดที่ขึ้นต้นด้วย '<index>.0,20YY-') -> ทุกอย่างก่อนหน้าสมบูรณ์
    size = path.stat().st_size
    with open(path, "rb") as f:
        f.seek(max(0, size - tail))
        buf = f.read()
    m = list(re.finditer(rb"\n\d+\.0,20\d\d-\d\d-\d\d ", buf))
    if not m:
        raise RuntimeError("หา record boundary ท้ายไฟล์ไม่เจอ")
    return max(0, size - tail) + m[-1].start() + 1


def scan_fnspid(sp_tickers, chunksize=200_000):
    if FNSPID_CACHE.exists() and FNSPID_META.exists():
        return pd.read_parquet(FNSPID_CACHE), json.loads(FNSPID_META.read_text())
    have = RAW_FNSPID.stat().st_size if RAW_FNSPID.exists() else 0
    if have == 0:
        raise FileNotFoundError(f"ไม่พบ {RAW_FNSPID}\nดาวน์โหลด: curl -L -C - -o \"{RAW_FNSPID}\" {FNSPID_URL}")
    partial = have < FNSPID_SIZE
    log(f"[fnspid] อ่าน {RAW_FNSPID.name} ({have / 1e9:.1f}/{FNSPID_SIZE / 1e9:.1f} GB"
        + (", ยังโหลดไม่ครบ -> อ่านเท่าที่มี" if partial else "") + ")")
    keep = set(sp_tickers) | {k for k, v in RENAMES.items() if v}
    cols = ["Date", "Article_title", "Stock_symbol", "Url", "Publisher", "Author", "Article", "Lsa_summary"]
    fh = open(RAW_FNSPID, "rb")
    if partial:                          # อ่านถึง record สุดท้ายที่สมบูรณ์เท่านั้น (ไม่ให้ chunk สุดท้ายพังทั้ง chunk)
        fh = io.BufferedReader(_Limited(fh, _last_record_start(RAW_FNSPID)), buffer_size=1 << 20)
    reader = pd.read_csv(fh, chunksize=chunksize, dtype=str, usecols=cols, on_bad_lines="skip",
                         encoding="utf-8", encoding_errors="replace")
    parts, n_read, last_sym, t0 = [], 0, None, time.time()
    while True:
        try:
            chunk = next(reader)
        except StopIteration:
            break
        except (pd.errors.ParserError, ValueError) as e:
            if partial:                     # ไฟล์ที่ยังโหลดไม่ครบจบกลาง record
                log(f"  [fnspid] หยุดที่ท้ายไฟล์ที่โหลดไม่ครบ: {str(e)[:80]}")
                break
            raise
        n_read += len(chunk)
        sym = chunk["Stock_symbol"].str.upper().str.strip()
        if sym.notna().any():
            last_sym = sym.dropna().iloc[-1]
        m = sym.isin(keep) & chunk["Date"].str[:4].isin(FNSPID_YEARS)
        if m.any():
            c = chunk[m].copy()
            c["ticker"] = sym[m].map(lambda s: RENAMES.get(s, s))
            c["Article"] = c["Article"].str.slice(0, C.article_chars)
            parts.append(c[c["ticker"].notna()].drop(columns="Stock_symbol").assign(symbol_raw=sym[m]))
        if (n_read // chunksize) % 10 == 0:
            log(f"  [fnspid] อ่าน {n_read / 1e6:.1f}M แถว | เก็บ {sum(len(p) for p in parts):,} | "
                f"ล่าสุด {last_sym} ({time.time() - t0:.0f}s)")
    df = pd.concat(parts, ignore_index=True)
    meta = {"partial": partial, "bytes": have, "rows_read": n_read, "last_symbol": last_sym if partial else None}
    if partial:   # ticker สุดท้ายอาจถูกตัดกลาง -> ทิ้งตัวนั้นและทุกตัวหลังจากนั้น
        df = df[df["symbol_raw"] < last_sym]
    df = df.drop(columns="symbol_raw").reset_index(drop=True)
    df.to_parquet(FNSPID_CACHE, index=False)
    FNSPID_META.write_text(json.dumps(meta))
    log(f"[fnspid] cache {len(df):,} แถว ({df['ticker'].nunique()} หุ้น) -> {FNSPID_CACHE.name} "
        f"| {time.time() - t0:.0f}s | {meta}")
    return df, meta


fn_raw, fn_meta = scan_fnspid(sp500["ticker"])


def fnspid_covered(ticker: str) -> bool:
    # หุ้นนี้อยู่ในส่วนของไฟล์ที่อ่านครบหรือไม่ (ไฟล์เรียงตาม ticker)
    if not fn_meta["partial"]:
        return True
    names = [ticker] + [k for k, v in RENAMES.items() if v == ticker]
    return any(n < fn_meta["last_symbol"] for n in names)


print(fn_meta)
print(f"{len(fn_raw):,} แถว | {fn_raw['ticker'].nunique()} หุ้น S&P 500")

### 1.2 ตรวจเวลาเผยแพร่และความหนาแน่นของข่าว
ถ้าเวลาเป็น `00:00:00` = มีแค่วันที่ → ใช้กฎ 16:00 ไม่ได้ (ใช้วันที่ตามที่มี และบันทึกเป็นข้อจำกัด)

In [ ]:
fn_ts = pd.to_datetime(fn_raw["Date"].str.replace(" UTC", "", regex=False), utc=True, errors="coerce")
fn_raw["published_utc"] = fn_ts
fn_raw["date_only"] = (fn_ts.dt.hour == 0) & (fn_ts.dt.minute == 0) & (fn_ts.dt.second == 0)
fn_raw = fn_raw.dropna(subset=["published_utc"])
fn_raw["year"] = fn_raw["published_utc"].dt.year

print("สัดส่วนข่าวที่ไม่มีเวลา (date only) รายปี:")
display(fn_raw.groupby("year")["date_only"].agg(["size", "mean"]).rename(columns={"size": "n", "mean": "date_only"}))
timed = fn_raw[~fn_raw["date_only"]]
if len(timed):
    print("ชั่วโมงเผยแพร่ (เวลานิวยอร์ก) ของข่าวที่มีเวลา:")
    display(timed["published_utc"].dt.tz_convert(NY).dt.hour.value_counts().sort_index().to_frame("n").T)
print("จำนวนข่าวต่อหุ้น (2021-2023):")
display(fn_raw.groupby("ticker").size().describe().to_frame("articles/ticker").T)

### 1.3 เลือกหุ้น
- มูลค่าซื้อขายเฉลี่ยปี 2020 (ก่อนช่วงข่าว) → `qcut` 3 กลุ่ม → สุ่มกลุ่มละ `n_per_bucket` แบบ round-robin ข้ามอุตสาหกรรม
- ตัดหุ้นที่ FNSPID ไม่ครอบคลุม (ไฟล์ไม่ครบ) และหุ้นที่มีข่าวน้อยกว่า `min_articles` (ไม่มีข่าวให้วัด)

In [ ]:
def select_universe():
    a, b = C.universe_ref
    dv_path = CACHE / f"sp500_dollarvol_{a.replace('-', '')}_{b.replace('-', '')}.parquet"
    if dv_path.exists():
        dv = pd.read_parquet(dv_path)
    else:
        log(f"[universe] ดึงราคา S&P 500 {a} -> {b} เพื่อจัดกลุ่มขนาด")
        px = download_prices(sp500["ticker"], a, (pd.Timestamp(b) + pd.Timedelta(days=1)).strftime("%Y-%m-%d"))
        dv = px.assign(dv=px["close"] * px["volume"]).groupby("ticker")["dv"].mean().rename("dollar_vol").reset_index()
        dv.to_parquet(dv_path, index=False)
    sp = sp500.merge(dv, on="ticker", how="inner").dropna(subset=["dollar_vol"])
    n_news = fn_raw.groupby("ticker").size()
    sp["n_news"] = sp["ticker"].map(n_news).fillna(0).astype(int)
    sp["covered"] = sp["ticker"].map(fnspid_covered)
    log(f"[universe] S&P 500 มีราคาปี 2020: {len(sp)} | FNSPID ไม่ครอบคลุม: {(~sp['covered']).sum()} "
        f"| ข่าว < {C.min_articles}: {(sp['covered'] & (sp['n_news'] < C.min_articles)).sum()}")
    sp = sp[sp["covered"] & (sp["n_news"] >= C.min_articles)].copy()
    sp["size_rank"] = pd.qcut(sp["dollar_vol"].rank(method="first"), 3, labels=[0, 1, 2]).astype(int)
    rng = np.random.default_rng(C.universe_seed)
    picks = []
    for _, g in sp.groupby("size_rank"):
        by_sector = {s: list(rng.permutation(d["ticker"].to_numpy())) for s, d in g.groupby("sector")}
        chosen = []
        while len(chosen) < min(C.n_per_bucket, len(g)):
            for s in sorted(by_sector, key=lambda s: rng.random()):
                if by_sector[s] and len(chosen) < C.n_per_bucket:
                    chosen.append(by_sector[s].pop())
        picks += chosen
    return sp[sp["ticker"].isin(picks)].sort_values(["size_rank", "ticker"]).reset_index(drop=True)


universe = select_universe()
universe.to_parquet(P("universe.parquet"), index=False)
TICKERS = universe["ticker"].tolist()
print(len(universe), "หุ้น")
display(universe.groupby(["sector", "size_rank"]).size().unstack(fill_value=0))

### 1.4 ทำความสะอาดข่าว → `news_clean.parquet`

In [ ]:
def build_news(raw: pd.DataFrame) -> pd.DataFrame:
    r = raw[raw["ticker"].isin(TICKERS)]
    df = pd.DataFrame({
        "article_id": "fns" + r["Url"].fillna(r["Article_title"]).map(short_hash),
        "ticker": r["ticker"],
        "published_utc": r["published_utc"],
        "date_only": r["date_only"],
        "title": r["Article_title"].fillna("").str.strip(),
        "summary_src": r["Lsa_summary"],
        "content": r["Article"],
        "source": r["Publisher"],
        "url": r["Url"],
        "author": r["Author"],
    })
    df = df[df["title"].str.len() > 0]
    df["published_ny"] = df["published_utc"].dt.tz_convert(NY)
    df["date_ny"] = np.where(df["date_only"], df["published_utc"].dt.strftime("%Y-%m-%d"),
                             df["published_ny"].dt.strftime("%Y-%m-%d"))
    df = df[(df["date_ny"] >= C.news_start) & (df["date_ny"] <= C.news_end)]
    n0 = len(df)
    df = df.sort_values("published_utc").drop_duplicates(["ticker", "article_id"])
    df = df[df["url"].isna() | ~df.duplicated(["ticker", "url"])]
    df["_nt"] = df["title"].map(norm_title)
    df = df.drop_duplicates(["ticker", "_nt", "date_ny"]).drop(columns="_nt")
    n1 = len(df)
    df = df.sample(frac=1, random_state=C.seed)
    df = df[df.groupby(["ticker", "date_ny"]).cumcount() < C.max_per_ticker_day]
    df["trade_date"] = to_trade_date(df["published_utc"], df["date_only"])
    df["row_id"] = df["article_id"] + ":" + df["ticker"]
    df = df.sort_values(["published_utc", "ticker"]).reset_index(drop=True)
    log(f"[clean] {n0:,} แถวในช่วง -> ลบซ้ำ {n1:,} -> จำกัด {C.max_per_ticker_day}/หุ้น/วัน {len(df):,} | "
        f"ไม่มีเวลา {df['date_only'].mean():.1%}")
    return df


news = build_news(fn_raw)
news.to_parquet(P("news_clean.parquet"), index=False)
print(f"{len(news):,} แถว | {news['article_id'].nunique():,} บทความ | {news['ticker'].nunique()} หุ้นมีข่าว")
display(news.groupby(news["trade_date"].dt.year).size().to_frame("n").T)
display(news[["ticker", "published_ny", "date_only", "trade_date", "source", "title"]].head(8))

### 1.5 ราคา → `prices.parquet`

In [ ]:
px_path, px_meta = P("prices.parquet"), P("prices_tickers.json")
want = sorted(TICKERS + BENCHMARKS)
if px_path.exists() and px_meta.exists() and json.loads(px_meta.read_text()) == want:
    prices = pd.read_parquet(px_path)          # cache ใช้ได้เมื่อรายชื่อหุ้นไม่เปลี่ยน
else:
    prices = download_prices(want, C.price_start, C.price_end)
    prices.to_parquet(px_path, index=False)
    px_meta.write_text(json.dumps(want))
missing = sorted(set(TICKERS + BENCHMARKS) - set(prices["ticker"]))
print(f"{prices['ticker'].nunique()} ticker | {prices['date'].min():%Y-%m-%d} -> {prices['date'].max():%Y-%m-%d}"
      + (f" | ไม่มีราคา: {missing}" if missing else " | ดึงราคาได้ครบทุกตัว"))

---
# ขั้นตอนที่ 2: ติดป้ายข่าวด้วย LLM (วันที่ 2–3)

ไม่ทำนักเรียน (distilroberta) — ให้ LLM ติดป้ายทุกแถวในการเรียกครั้งเดียว รวมการคัดกรองของขั้นตอนที่ 1:
`relevant, important, sentiment (−2..+2), summary, primary_tag, secondary_tag, scheduled, quantified, rumor`

- JSON schema ล็อก enum · temperature 0 · ตัวอย่าง 5 บทความ · **สั่งให้ตัดสินจากเนื้อข่าวเท่านั้น ห้ามใช้ความรู้อนาคต/ทำนายราคา**
- ส่งซ้ำถ้า JSON ผิดรูปแบบ · cache ทีละแถวใน `_cache/llm/<prompt_version>__<model>.jsonl` (หยุดกลางทางแล้วทำต่อได้, รันซ้ำไม่เสียเงินซ้ำ)
- บันทึก `label_source` (โมเดล) และ `prompt_version` ทุกแถว

In [ ]:
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
TagT = Literal[tuple(TAG_IDS)]
Tag2T = Literal[tuple(TAG_IDS + ["none"])]


class NewsLabel(BaseModel):
    model_config = ConfigDict(extra="forbid")
    reasoning: str = Field(description="One short sentence: the new event about THIS company, if any")
    relevant: bool = Field(description="Article is mainly about this company (see Rules)")
    important: bool = Field(description="Reports a concrete new company event an investor would care about (see Rules)")
    sentiment: Literal[-2, -1, 0, 1, 2] = Field(description="Tone of the event for THIS company's shareholders, -2 .. +2")
    summary: str = Field(description="One sentence (<= 30 words) summarizing the event")
    primary_tag: TagT = Field(description="Event type")
    secondary_tag: Tag2T = Field(description="Second event type, or 'none'")
    scheduled: bool = Field(description="Event was pre-scheduled (earnings date, dividend date, FDA date, investor day)")
    quantified: bool = Field(description="Reports company fundamentals as numbers (see Rules)")
    rumor: bool = Field(description="Unconfirmed: sources say, reportedly, in talks, considering")


def strict_schema(model) -> dict:
    schema = model.model_json_schema()
    defs = schema.pop("$defs", {})
    drop = {"title", "default", "minimum", "maximum", "minItems", "maxItems"}

    def walk(node):
        if isinstance(node, dict):
            if "$ref" in node:
                return walk(defs[node["$ref"].split("/")[-1]])
            node = {k: walk(v) for k, v in node.items() if k not in drop}
            if node.get("type") == "object":
                node["additionalProperties"] = False
                node["required"] = list(node.get("properties", {}))
            return node
        if isinstance(node, list):
            return [walk(x) for x in node]
        return node

    return walk(schema)


RESPONSE_FORMAT = {"type": "json_schema", "json_schema": {"name": "news_label", "strict": True, "schema": strict_schema(NewsLabel)}}

TAG_DEFS = {
    "earnings_guidance": "Quarterly/annual results, beats/misses, earnings call transcripts, guidance raised/cut/reaffirmed.",
    "analyst": "Analyst upgrades/downgrades, initiations, price target or estimate changes.",
    "dividend_buyback": "Dividends declared/raised/cut/special; share buybacks.",
    "mna": "Mergers, acquisitions, divestitures, spin-offs, takeover bids, merger talks, asset sales.",
    "legal_regulatory": "Lawsuits, settlements, investigations, fines, antitrust, regulator decisions (FDA/FTC/SEC/DOJ/EU), drug approvals/rejections, ordered recalls.",
    "product_partnership": "Product launches, contracts, customer wins, partnerships, JVs, supply deals, trial results.",
    "management": "Executive/board changes, governance, exec pay, insider trades, layoffs/restructuring.",
    "macro_other": "Other concrete company events: operations, supply chain, capacity, capital raising, credit ratings, or an industry/macro event tied to this company.",
    "price_commentary": "JUNK, relevant, no new event: this stock's price move, technicals, options, valuation opinion, 'should you buy'.",
    "promotional": "JUNK: every relevant=false article; also ads, sponsored content, stock-picking lists, newsletters.",
}

FEW_SHOT = [
    ({"ticker": "XYZ", "company": "Xyz Corp", "title": "Xyz Corp Q3 EPS $1.42 Beats $1.30 Estimate, Sales $8.1B Beat; Raises FY Guidance",
      "text": "Xyz Corp reported third-quarter earnings of $1.42 per share, above the consensus of $1.30. Revenue rose 9% to $8.1 billion. The company raised its full-year EPS outlook to $5.60-$5.70."},
     {"reasoning": "Scheduled Q3 results beat estimates; guidance raised.", "relevant": True, "important": True,
      "sentiment": 2, "summary": "Xyz Corp Q3 EPS and revenue beat estimates and full-year guidance was raised.",
      "primary_tag": "earnings_guidance", "secondary_tag": "none", "scheduled": True, "quantified": True, "rumor": False}),
    ({"ticker": "XYZ", "company": "Xyz Corp", "title": "Consumer Stocks Sector Update: Lmn, Xyz Higher",
      "text": "Consumer stocks were rising Tuesday, with the sector ETF up 0.6%. In company news, Lmn Foods reported Q2 EPS of $0.88, above estimates. Xyz Corp rose 1.9% after announcing a $2B buyback."},
     {"reasoning": "Sector recap headline; Rule 1 applies despite the buyback.", "relevant": False, "important": False,
      "sentiment": 0, "summary": "Xyz Corp is mentioned in a consumer sector recap.",
      "primary_tag": "promotional", "secondary_tag": "none", "scheduled": False, "quantified": False, "rumor": False}),
    ({"ticker": "ABC", "company": "Abc Inc", "title": "Barclays Maintains Overweight on Abc Inc, Price Target $120",
      "text": "Barclays analyst kept an Overweight rating on Abc Inc and maintained a $120 price target."},
     {"reasoning": "Rating and price target unchanged.", "relevant": True, "important": False,
      "sentiment": 0, "summary": "Barclays maintains Overweight on Abc Inc with a $120 price target.",
      "primary_tag": "analyst", "secondary_tag": "none", "scheduled": False, "quantified": True, "rumor": False}),
]


def _user_text(x: dict, chars: int) -> str:
    text = (x.get("text") or "").strip()[:chars] or "(no body text)"
    return (f"Target company: {x.get('company') or x['ticker']} (ticker {x['ticker']})\n"
            f"Published: {x.get('published', '-')}\nSource: {x.get('source', '-')}\n"
            f"Headline: {x['title']}\nBody:\n{text}")


def build_system_prompt() -> str:
    tags = "\n".join(f"- {k}: {v}" for k, v in TAG_DEFS.items())
    shots = "\n\n".join(f"Input:\n{_user_text(i, 600)}\nOutput:\n{json.dumps(o)}" for i, o in FEW_SHOT)
    return f"""Label ONE financial news article for ONE target company (US stock event study). Return JSON per the schema.
Use only the article text: no knowledge of later events, prices or outcomes. Never predict prices.

## Event types (primary_tag / secondary_tag)
{tags}

## Rules
1. Judge relevant from the headline. relevant=false: market/sector recaps ("Sector Update", "Stocks rally"), movers lists, multi-company reports ("Ex-Dividend Reminder"), headlines naming several firms, macro news, or another firm's story; even if the body has news on this company.
   Then primary_tag=promotional and important=false.
2. important=false for junk tags and analyst "maintains/reiterates" without a rating/target change.
3. sentiment = tone of the NEW event for shareholders, not the stock move. 0 = neutral/unclear (e.g. unchanged dividend).
4. A price move explained by a new event ("shares fall after FDA rejection") gets that event's tag.
5. If two types fit: primary_tag = root event, secondary_tag = the other (else "none").
6. quantified=true only for company fundamentals: EPS, revenue, guidance, dividend, price target, deal value. Stock prices and % moves do not count.
7. rumor=true only for unconfirmed news.
8. Ignore ads and boilerplate in the body.

## Examples
{shots}"""


SYSTEM_PROMPT = build_system_prompt()


def build_messages(row: dict, chars: int) -> list:
    body = row.get("content") or row.get("summary_src") or ""
    if row.get("summary_src") and row.get("content"):
        body = f"{row['summary_src']}\n{row['content']}"
    x = {"ticker": row["ticker"], "company": row.get("company"), "title": row["title"], "text": body,
         "published": pd.Timestamp(row["published_ny"]).strftime("%Y-%m-%d %H:%M") if not row.get("date_only")
         else str(row.get("date_ny")), "source": row.get("source") or "-"}
    system = ([{"type": "text", "text": SYSTEM_PROMPT, "cache_control": {"type": "ephemeral"}}]     # prompt caching: ข้อความเดิม
              if getattr(C, "prompt_cache", False) else SYSTEM_PROMPT)
    return [{"role": "system", "content": system}, {"role": "user", "content": _user_text(x, chars)}]


print(SYSTEM_PROMPT[:1500], "...")

In [ ]:
_SESSION = None
_NO_TEMPERATURE = set()      # โมเดลที่ไม่รับ temperature -> ส่งโดยไม่มี temperature


def _session():
    global _SESSION
    if _SESSION is None:
        key = os.environ.get("OPENROUTER_API_KEY")
        assert key, "ใส่ OPENROUTER_API_KEY ใน Research/.env ก่อน"
        _SESSION = requests.Session()
        _SESSION.headers.update({"Authorization": f"Bearer {key}", "Content-Type": "application/json",
                                 "X-Title": "KMUTT Senior Project - Buy the Rumor"})
    return _SESSION


def chat(messages, model, max_tokens=500, retries=4, response_format=None):
    body = {"model": model, "messages": messages, "temperature": 0, "max_tokens": max_tokens,
            "response_format": response_format or RESPONSE_FORMAT, "provider": {"require_parameters": True}, "usage": {"include": True}}
    if model in _NO_TEMPERATURE:
        body.pop("temperature")
    for attempt in range(retries):
        try:
            r = _session().post(OPENROUTER_URL, json=body, timeout=120)
        except requests.RequestException:
            if attempt < retries - 1:
                time.sleep(2 ** attempt * 2)
                continue
            raise
        if r.status_code == 404 and "temperature" in body and "No endpoints found" in r.text:
            _NO_TEMPERATURE.add(model)
            body.pop("temperature")
            continue
        if r.status_code in (408, 429, 500, 502, 503, 504) and attempt < retries - 1:
            time.sleep(2 ** attempt * 2)
            continue
        if not r.ok:
            raise RuntimeError(f"OpenRouter {r.status_code}: {r.text[:300]}")
        data = r.json()
        if "error" in data:
            raise RuntimeError(f"OpenRouter error: {data['error']}")
        return data["choices"][0]["message"]["content"] or "", data.get("usage", {})
    raise RuntimeError("OpenRouter: retries exhausted")


def label_one(row: dict, model: str, chars: int, tries=3):
    err = None
    for _ in range(tries):                # ส่งซ้ำถ้า JSON ผิดรูปแบบ / ป้ายไม่อยู่ใน enum
        text, usage = chat(build_messages(row, chars), model)
        try:
            m = re.search(r"\{.*\}", text.strip(), re.S)
            x = NewsLabel.model_validate(json.loads(m.group(0) if m else text))
            if x.secondary_tag == x.primary_tag:
                x.secondary_tag = "none"
            return x, usage
        except (ValidationError, json.JSONDecodeError) as e:
            err = e
    raise RuntimeError(f"invalid JSON after {tries} tries: {str(err)[:200]}")


def cache_path(model: str, prompt_version=None) -> Path:
    p = CACHE / "llm" / f"{prompt_version or C.prompt_version}__{model.replace('/', '_').replace(':', '_')}.jsonl"
    p.parent.mkdir(parents=True, exist_ok=True)
    return p


def load_cache(model: str) -> dict:
    p = cache_path(model)
    if not p.exists():
        return {}
    out = {}
    for line in p.read_text(encoding="utf-8").splitlines():
        if line.strip():
            r = json.loads(line)
            out[r["row_id"]] = r
    return out


def cost_per_row(model: str) -> float:
    c = [r["cost"] for r in load_cache(model).values() if r.get("cost")]
    return float(np.mean(c)) if c else float("nan")


def label_rows(df: pd.DataFrame, model: str, limit=None, workers=None) -> pd.DataFrame:
    # ติดป้ายทุกแถวที่ยังไม่อยู่ใน cache (บันทึกทีละแถว) แล้วคืนป้ายของทุกแถวใน df ที่มีป้ายแล้ว
    done = load_cache(model)
    todo = [r for r in df.to_dict("records") if r["row_id"] not in done]
    if limit is not None:
        todo = todo[:limit]
    log(f"[llm] {model} | prompt {C.prompt_version} | cached {len(done)} | todo {len(todo)}")
    cost, fails, lock, t0 = 0.0, 0, threading.Lock(), time.time()
    with open(cache_path(model), "a", encoding="utf-8") as f, ThreadPoolExecutor(workers or C.llm_workers) as ex:
        futs = {ex.submit(label_one, r, model, C.content_chars): r for r in todo}
        for i, fut in enumerate(as_completed(futs), 1):
            r = futs[fut]
            try:
                x, usage = fut.result()
            except Exception as e:          # noqa: BLE001 — บันทึกแล้วไปต่อ
                fails += 1
                log(f"  ! {r['row_id']} {r['title'][:60]!r}: {str(e)[:160]}")
                continue
            rec = {"row_id": r["row_id"], "model": model, "prompt_version": C.prompt_version,
                   "cost": float(usage.get("cost") or 0), **x.model_dump()}
            with lock:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                f.flush()
                done[r["row_id"]] = rec
                cost += rec["cost"]
            if i % 1000 == 0:
                log(f"  {i}/{len(todo)} cost=${cost:.3f} ({time.time() - t0:.0f}s)")
    if todo:
        log(f"[llm] เสร็จ {len(todo) - fails}/{len(todo)} | fail {fails} | cost ${cost:.4f} "
            f"(${cost / max(1, len(todo) - fails):.6f}/แถว) | {time.time() - t0:.0f}s")
    return pd.DataFrame([done[k] for k in df["row_id"] if k in done])


news = pd.read_parquet(P("news_clean.parquet")).merge(
    pd.read_parquet(P("universe.parquet"))[["ticker", "name"]].rename(columns={"name": "company"}), on="ticker", how="left")
print(len(news), "แถวรอติดป้าย")

### 2.0 ตัวกรองหัวข่าว (ลดค่า LLM)
ข่าวที่หัวข่าวตรงรูปแบบขยะชัดเจน (Sector Update, ETF inflow/outflow, option activity, US STOCKS-, Validea/Guru report, ลิสต์ "3 Stocks to Buy", "Reasons to Buy" ฯลฯ) → ติดป้ายขยะเลยโดยไม่ส่ง LLM · ข่าวรวมตลาดที่มีชื่อบริษัทเป้าหมายในหัวข่าวจะไม่ถูกกรอง
- เกณฑ์: ข่าวที่คนเก็บใน `hand_labels.csv` ต้องถูกกรอง **0** ข่าว · ดูตัวอย่างที่ถูกกรองกฎละ 3 ข่าว
- แก้ / ลบกฎใน `TITLE_RULES` ได้ · ปิดทั้งหมดด้วย `title_filter=False` ใน config
- แถวที่ถูกกรองบันทึก `label_source = "title_filter"` และ `reasoning = "title_filter:<กฎ>"`

In [ ]:
# (ชื่อกฎ, ป้ายขยะที่ใส่ให้, regex บนหัวข่าวตัวพิมพ์เล็ก, ยกเว้นถ้าหัวข่าวมีชื่อ/ticker ของบริษัทเป้าหมาย)
TITLE_RULES = [
    ("quant_screen",  "promotional",      r"guru fundamental report|validea|detailed fundamental analysis|peter lynch|quantitative stock analysis", False),
    ("sector_update", "promotional",      r"\bsector update\b", False),
    ("most_active",   "promotional",      r"(?:after hours|after-hours|pre-market|premarket) most active", False),
    ("etf_flow",      "promotional",      r"(?:notable|noteworthy) etf (?:inflow|outflow)", False),
    ("options",       "promotional",      r"(?:notable|noteworthy) \w+ option activity|unusual (?:put|call) option|put and call options|implied volatility", False),
    ("market_recap",  "promotional",      r"^us stocks[- ]|^buzz-u\.s\. stocks|^stock market news for", True),
    ("zacks_digest",  "promotional",      r"zacks analyst blog|zacks industry outlook|zacks investment ideas|top (?:analyst|stock|research) reports for", False),
    ("listicle",      "promotional",      r"^(?:top )?\d+ [\w\s'-]{0,40}\bstocks?\b|dividend aristocrats|^if you invested \$|most undervalued", False),
    ("buy_pitch",     "promotional",      r"should you invest in|here's why you should (?:buy|add|hold|retain|consider)|reasons to (?:buy|own|add)|a strong etf right now", False),
    ("listicle2",     "promotional",      r"^the \d+ best|^\d+% of warren buffett|^how to find (?:the )?(?:best|top)|investors heavily search|zacks\.com featured highlights", False),
    ("multi_report",  "promotional",      r"^s&p \d+ movers|^company news for|^daily dividend report|options begin trading", False),
    ("pitch2",        "promotional",      r"^should you buy\b", False),
]
TITLE_RULE_TAG = {name: tag for name, tag, _, _ in TITLE_RULES}


def _short_name(name) -> str:
    w = re.sub(r"[^a-z0-9 ]", " ", str(name or "").lower()).split()
    return w[0] if w and len(w[0]) >= 3 else ""


def title_filter_rule(df: pd.DataFrame) -> pd.Series:
    # ชื่อกฎแรกที่หัวข่าวตรง (None = ไม่ถูกกรอง ส่งให้ LLM)
    t = df["title"].fillna("").str.lower()
    out = pd.Series(None, index=df.index, dtype=object)
    for name, _, rx, exc in TITLE_RULES:
        hit = out.isna() & t.str.contains(rx, regex=True)
        if exc and hit.any():               # ข่าวรวมตลาดที่ระบุชื่อบริษัทเป้าหมายในหัวข่าว -> ไม่กรอง (อาจเป็นข่าวของบริษัทนี้)
            names = (df["company"] if "company" in df else pd.Series("", index=df.index)).map(_short_name)
            own = [bool((n and re.search(r"\b" + re.escape(n), s)) or re.search(r"\b" + re.escape(str(k).lower()) + r"\b", s))
                   for s, n, k in zip(t[hit], names[hit], df.loc[hit, "ticker"])]
            hit.loc[hit] = ~np.array(own, dtype=bool)
        out[hit] = name
    return out


def title_filter_labels(df: pd.DataFrame, rule: pd.Series) -> pd.DataFrame:
    # ป้ายของแถวที่ถูกกรอง (ไม่เรียก LLM): ขยะ ไม่เกี่ยวข้อง ไม่สำคัญ เป็นกลาง
    d = df.loc[rule.notna(), ["row_id", "title"]]
    r = rule[rule.notna()]
    return pd.DataFrame({
        "row_id": d["row_id"].to_numpy(), "model": "title_filter", "prompt_version": "title_filter",
        "cost": 0.0, "reasoning": ("title_filter:" + r).to_numpy(), "relevant": False, "important": False,
        "sentiment": 0, "summary": d["title"].to_numpy(), "primary_tag": r.map(TITLE_RULE_TAG).to_numpy(),
        "secondary_tag": "none", "scheduled": False, "quantified": False, "rumor": False,
    })


TF_RULE = title_filter_rule(news) if C.title_filter else pd.Series(None, index=news.index, dtype=object)
NEWS_LLM = news[TF_RULE.isna()]            # แถวที่ต้องส่ง LLM
print(f"ตัวกรองหัวข่าว: {'เปิด' if C.title_filter else 'ปิด'} | กรองออก {TF_RULE.notna().sum():,}/{len(news):,} "
      f"({TF_RULE.notna().mean():.1%}) | ส่ง LLM {len(NEWS_LLM):,} แถว")
display(TF_RULE.value_counts().rename("n").to_frame().T)

# ตรวจกับ hand label: ต้องไม่ทิ้งข่าวที่คนเลือกเก็บ (relevant & important)
if P("hand_labels.csv").exists():
    _h = pd.read_csv(P("hand_labels.csv"))
    _h["company"] = _h["ticker"].map(news.drop_duplicates("ticker").set_index("ticker")["company"])
    _hr = title_filter_rule(_h)
    _t = lambda s: s.astype(str).str.lower().isin(["1", "1.0", "true", "yes", "y"])     # noqa: E731
    _keep, _junk = _t(_h["hand_relevant"]) & _t(_h["hand_important"]), _h["hand_primary_tag"].isin(JUNK_TAGS)
    print(f"hand label: กรองออก {_hr.notna().sum()}/{len(_h)} | ข่าวที่คนเก็บแต่ถูกกรอง {int((_hr.notna() & _keep).sum())} (ต้องเป็น 0) "
          f"| จับข่าวขยะได้ {int((_hr.notna() & _junk).sum())}/{int(_junk.sum())}")
    _bad = _h[_hr.notna() & ~_junk]
    if len(_bad):
        display(_bad[["ticker", "title", "hand_primary_tag", "hand_relevant", "hand_important"]].assign(rule=_hr[_bad.index]))

# สุ่มดูหัวข่าวที่ถูกกรอง กฎละ 3 ข่าว (ให้ A ตรวจว่าไม่มีข่าวจริงหลุดมา)
_s = news.assign(rule=TF_RULE).dropna(subset=["rule"])
display(_s.groupby("rule").sample(n=3, replace=False, random_state=0)[["rule", "ticker", "title"]] if _s["rule"].value_counts().min() >= 3
        else _s.groupby("rule").head(3)[["rule", "ticker", "title"]])

### 2.1 ทดลอง 200 บทความ → ประมาณค่าใช้จ่าย
ปรับ prompt 2–3 รอบ (แก้ `TAG_DEFS` / `FEW_SHOT` แล้ว **bump `prompt_version`** ใน config) แล้วตรึงไว้ก่อนรันชุดเต็ม

In [ ]:
pilot = NEWS_LLM.sample(min(200, len(NEWS_LLM)), random_state=1)
pilot_labs = label_rows(pilot, C.model)
cpr = cost_per_row(C.model)
todo_n = (~NEWS_LLM["row_id"].isin(load_cache(C.model))).sum()     # ไม่นับแถวที่ตัวกรองหัวข่าวตัดแล้ว
est = cpr * todo_n
print(f"ค่าใช้จ่ายเฉลี่ย ${cpr:.6f}/แถว | ยังไม่ติดป้าย {todo_n:,} แถว -> ประมาณ ${est:.2f} (งบ ${C.llm_budget_usd})")
p = pilot.merge(pilot_labs, on="row_id")
display(pd.crosstab(p["primary_tag"], p["sentiment"], margins=True))
display(p[["ticker", "title", "relevant", "important", "sentiment", "primary_tag", "rumor", "summary"]].head(10))

### 2.2 ตรวจคุณภาพด้วยมือ (150 บทความ: tune 75 / test 75) — ติดป้าย 2 คน
1. รัน cell ถัดไป → ได้ `hand_labels_a.csv` (A กรอก) และ `hand_labels_b.csv` (B กรอก) — บทความชุดเดียวกัน **ต่างคนต่างกรอก ห้ามดูของอีกคน / ห้ามดูป้าย LLM** ใช้คำนิยามใน `TAG_DEFS`
2. ทั้งสองคนกรอกครบแล้วรันอีกครั้ง → ตารางความตรงกันระหว่างคน (% ตรงกัน, Cohen's kappa) + คู่ป้ายที่สับสน และสร้าง `hand_labels.csv` (ช่องที่ตรงกันเติมให้แล้ว ช่องที่ไม่ตรงเว้นว่าง ดู `a_*` / `b_*` ประกอบ)
3. กรอกช่องว่างใน `hand_labels.csv` ร่วมกัน (คำตอบสุดท้าย) แล้วรันอีกครั้ง → LLM ติดป้ายชุดนี้ (~$0.14) และวัดผล — เป้าหมายป้ายหลักตรงกัน ≥ 85% บนชุด `test`

ไม่สร้างไฟล์ทับของเดิม: ถ้า `news` เปลี่ยน (เช่น โหลด FNSPID ใหม่) ต้องลบไฟล์ `hand_labels*.csv` ใน `btr_<MODE>/` เอง **ก่อน**เริ่มกรอก

In [ ]:
HAND_FIELDS = ["relevant", "important", "sentiment", "primary_tag", "scheduled", "quantified", "rumor"]
HAND_BOOL = ["relevant", "important", "scheduled", "quantified", "rumor"]
HAND_INFO = ["row_id", "ticker", "published_ny", "source", "title", "summary_src", "split"]
hand_path = lambda who: P(f"hand_labels_{who}.csv")      # noqa: E731  who = "a" | "b"


def make_hand_label_sheets(n=150, seed=0):
    # สร้าง hand_labels_a.csv / hand_labels_b.csv: บทความชุดเดียวกัน แยกไฟล์ให้แต่ละคนกรอกโดยไม่เห็นคำตอบของอีกคน
    # ไม่เขียนทับไฟล์ที่มีอยู่ (ถ้ามีไฟล์เดียว จะสร้างอีกไฟล์จากแถวชุดเดียวกัน)
    pa, pb = hand_path("a"), hand_path("b")
    if pa.exists() and pb.exists():
        return pa, pb
    if pa.exists() or pb.exists():
        sheet = pd.read_csv(pa if pa.exists() else pb)[HAND_INFO]
    else:
        df = news.sample(min(n, len(news)), random_state=seed)
        sheet = df[HAND_INFO[:-1]].copy()
        sheet["split"] = ["tune" if i < len(sheet) // 2 else "test" for i in range(len(sheet))]
    for c in HAND_FIELDS:
        sheet[f"hand_{c}"] = ""
    for p in (pa, pb):
        if not p.exists():
            sheet.to_csv(p, index=False, encoding="utf-8-sig")
            log(f"เขียน {p}")
    log(f"primary_tag ใช้: {', '.join(TAG_IDS)} | ค่าจริง/เท็จ: 1/0 | sentiment: -2..2")
    return pa, pb


def _norm_hand(df):
    # แปลงค่าที่กรอก -> ชนิดเดียวกัน ช่องว่าง = NaN (ยังไม่กรอก)
    out = df[["row_id", "split"]].copy()
    for c in HAND_FIELDS:
        s = df[f"hand_{c}"].astype(str).str.strip().str.lower().replace({"nan": "", "none": ""})
        if c in HAND_BOOL:
            v = s.isin(["1", "1.0", "true", "yes", "y"]).astype(float)
        elif c == "sentiment":
            v = pd.to_numeric(s, errors="coerce")
        else:
            v = s.astype(object)
        out[c] = v.where(s != "")
    return out


def cohen_kappa(x, y) -> float:
    x, y = pd.Series(x).astype(str).to_numpy(), pd.Series(y).astype(str).to_numpy()
    if len(x) == 0:
        return np.nan
    po = (x == y).mean()
    cats = np.union1d(x, y)
    pe = sum((x == k).mean() * (y == k).mean() for k in cats)
    return 1.0 if pe == 1 else (po - pe) / (1 - pe)


def load_hand_pair() -> pd.DataFrame:
    a = _norm_hand(pd.read_csv(hand_path("a")))
    b = _norm_hand(pd.read_csv(hand_path("b")))
    return a.merge(b.drop(columns="split"), on="row_id", suffixes=("_a", "_b"))


def inter_annotator() -> pd.DataFrame:
    # ความตรงกันระหว่าง A กับ B รายช่อง (นับเฉพาะแถวที่ทั้งสองคนกรอกแล้ว)
    m = load_hand_pair()
    rows = []
    for c in HAND_FIELDS:
        g = m.dropna(subset=[f"{c}_a", f"{c}_b"])
        r = {"field": c, "n": len(g),
             "agree": (g[f"{c}_a"] == g[f"{c}_b"]).mean() if len(g) else np.nan,
             "kappa": cohen_kappa(g[f"{c}_a"], g[f"{c}_b"])}
        if c == "sentiment" and len(g):
            r["sign_agree"] = (np.sign(g[f"{c}_a"]) == np.sign(g[f"{c}_b"])).mean()
        rows.append(r)
    return pd.DataFrame(rows).round(3)


def tag_confusion() -> pd.DataFrame:
    # คู่ป้ายหลักที่ A กับ B ไม่ตรงกัน (ใช้หาป้ายที่นิยามไม่ชัด)
    m = load_hand_pair().dropna(subset=["primary_tag_a", "primary_tag_b"])
    d = m[m["primary_tag_a"] != m["primary_tag_b"]]
    return d.groupby(["primary_tag_a", "primary_tag_b"]).size().rename("n").sort_values(ascending=False).reset_index()


def make_consensus_sheet(overwrite=False):
    # hand_labels.csv = คำตอบสุดท้าย: ช่องที่ A == B เติมให้แล้ว, ช่องที่ไม่ตรงกันเว้นว่าง (ดูค่า a_* / b_* แล้วกรอกร่วมกัน)
    path = P("hand_labels.csv")
    if path.exists() and not overwrite:
        log(f"มี {path} แล้ว (ส่ง overwrite=True ถ้าต้องการสร้างใหม่ — คำตอบที่ตกลงไว้จะหาย)")
        return path
    raw_a = pd.read_csv(hand_path("a"))
    raw_b = pd.read_csv(hand_path("b")).set_index("row_id").loc[raw_a["row_id"]].reset_index()
    a, b = _norm_hand(raw_a), _norm_hand(raw_b)
    sheet = raw_a[HAND_INFO].copy()
    n_dis = 0
    for c in HAND_FIELDS:
        same = (a[c] == b[c]) & a[c].notna()
        n_dis += int((~same & a[c].notna() & b[c].notna()).sum())
        val = raw_a[f"hand_{c}"].astype(str).str.strip()
        sheet[f"hand_{c}"] = val.where(same, "")
        sheet[f"a_{c}"] = raw_a[f"hand_{c}"]
        sheet[f"b_{c}"] = raw_b[f"hand_{c}"]
    sheet["n_disagree"] = sheet[[f"hand_{c}" for c in HAND_FIELDS]].eq("").sum(axis=1)
    sheet.to_csv(path, index=False, encoding="utf-8-sig")
    log(f"เขียน {path} | ช่องที่ไม่ตรงกัน {n_dis} ช่อง ใน {(sheet['n_disagree'] > 0).sum()} แถว -> กรอก hand_* ที่ว่างร่วมกัน")
    return path


def evaluate_hand_labels(model=None) -> pd.DataFrame:
    # LLM เทียบกับคำตอบสุดท้าย (hand_labels.csv)
    hand = pd.read_csv(P("hand_labels.csv"))
    hand = hand[hand["hand_primary_tag"].notna() & (hand["hand_primary_tag"].astype(str).str.strip() != "")]
    if hand.empty:
        return pd.DataFrame()
    labs = pd.DataFrame(load_cache(model or C.model).values())
    if labs.empty:
        log("ยังไม่มีป้ายจาก LLM ใน cache (ตรวจ OPENROUTER_API_KEY ใน .env)")
        return pd.DataFrame()
    m = hand.merge(labs, on="row_id", how="inner")
    truthy = lambda s: s.astype(str).str.lower().isin(["1", "1.0", "true", "yes", "y"])     # noqa: E731
    rows = []
    for split, g in m.groupby("split"):
        r = {"split": split, "n": len(g), "primary_tag_acc": (g["hand_primary_tag"] == g["primary_tag"]).mean()}
        for c in HAND_BOOL:
            r[f"{c}_acc"] = (truthy(g[f"hand_{c}"]) == g[c].astype(bool)).mean()
        r["sentiment_sign_acc"] = (np.sign(pd.to_numeric(g["hand_sentiment"], errors="coerce")) == np.sign(g["sentiment"])).mean()
        rows.append(r)
    return pd.DataFrame(rows)


# 1) สร้างไฟล์ให้ A และ B กรอก (ไม่เสียเงิน)
print(make_hand_label_sheets())

# 2) เมื่อทั้งสองคนกรอกแล้ว: ความตรงกันระหว่างคน + คู่ป้ายที่สับสน
_filled = lambda who: pd.read_csv(hand_path(who))["hand_primary_tag"].notna().any()      # noqa: E731
if _filled("a") and _filled("b"):
    display(inter_annotator())
    display(tag_confusion())
    make_consensus_sheet()
else:
    print("รอ A และ B กรอก hand_labels_a.csv / hand_labels_b.csv")

# 3) เมื่อหาข้อยุติใน hand_labels.csv แล้ว: ให้ LLM ติดป้ายแถวชุดนี้ (~$0.14) แล้ววัดผล
#    เรียก LLM เมื่อกรอกช่อง hand_* ครบทุกแถวแล้วเท่านั้น
if P("hand_labels.csv").exists():
    _final = pd.read_csv(P("hand_labels.csv"))
    _blank = _final[[f"hand_{c}" for c in HAND_FIELDS]].isna().any(axis=1)
    if _blank.any():
        print(f"hand_labels.csv ยังมีช่องว่าง {int(_blank.sum())} แถว -> หาข้อยุติให้ครบก่อน (ยังไม่เรียก LLM)")
    else:
        hand_rows = news[news["row_id"].isin(_final["row_id"])]
        _ = label_rows(hand_rows, C.model)
        display(evaluate_hand_labels())
else:
    print("ยังไม่มี hand_labels.csv -> ข้ามการเรียก LLM")

### 2.2b รอบคัดกรอง (ลดค่าใช้จ่ายชุดเต็ม)
prompt สั้น (`screen_version`, cache แยก) ถามแค่ `keep` = ข่าวนี้อาจเป็นเหตุการณ์ใหม่ของบริษัทนี้ไหม (กฎเดียวกับข้อ 1–2 ของ v4, ไม่แน่ใจ → keep) · เฉพาะแถวที่ผ่านจึงส่งรอบเต็ม `rumor-v4` · แถวที่มีป้าย v4 ใน cache แล้วใช้ป้ายนั้นเสมอ · แถวที่ถูกตัด = `relevant=false` → `promotional` (`label_source = "screen"`)

cell นี้ **ตรวจก่อนรันชุดเต็ม** (เกณฑ์: ไม่ทิ้งข่าวที่ควรเก็บ) แล้วประมาณค่าใช้จ่ายรวม
- (ก) hand label: ข่าวที่คนเก็บต้องผ่าน **ทุกข่าว** (~$0.02)
- (ข) แถวที่มีป้าย v4 แล้ว ~1,800 ข่าว: นับข่าวที่ v4 เก็บแต่รอบคัดกรองตัด (~$0.3, ปิดได้ `screen_validate_v4=False`)
- ถ้าตัดข่าวที่ควรเก็บมากเกินไป → แก้ `SCREEN_PROMPT` แล้ว **เปลี่ยน `screen_version`** · ปิดรอบคัดกรองทั้งหมด `screen=False`

In [ ]:
class ScreenLabel(BaseModel):
    model_config = ConfigDict(extra="forbid")
    keep: bool = Field(description="True if the article may report a concrete new event mainly about the target company")


SCREEN_FORMAT = {"type": "json_schema", "json_schema": {"name": "news_screen", "strict": True, "schema": strict_schema(ScreenLabel)}}


class ScreenItem(BaseModel):
    model_config = ConfigDict(extra="forbid")
    id: int
    keep: bool


class ScreenBatch(BaseModel):
    model_config = ConfigDict(extra="forbid")
    results: list[ScreenItem]


SCREEN_BATCH_FORMAT = {"type": "json_schema", "json_schema": {"name": "news_screen_batch", "strict": True, "schema": strict_schema(ScreenBatch)}}

SCREEN_RULES = """keep=false ONLY for clear junk:
- market/sector recaps, movers lists, multi-company or ETF reports, macro news, or another firm's story,
  unless the headline names a new event of the target company itself
- stock price commentary with no new company event anywhere in the text, technicals, options activity,
  valuation opinions, "should you buy", stock-picking lists, ads
- analyst "maintains/reiterates" with no rating or price target change

keep=true for any new event of the target company, even inside a recap or a price-move story:
earnings or guidance, analyst rating/target change, dividend or buyback, M&A or activist/spin-off news,
legal/regulatory, product/contract/partnership, management change, insider buying/selling,
operations, financing, or an unconfirmed report of any of these.
Examples of keep=true: "US STOCKS-Futures slip on Apple's weak forecast" (target Apple);
"Why Hasbro stock popped today" when the body explains the move with a new company event."""

SCREEN_PROMPT = """Screen ONE financial news article for ONE target company. Return JSON {"keep": true|false}.
Use only the article text. This is a cheap first pass: when in doubt, keep=true.

""" + SCREEN_RULES       # ทีละข่าว (screen-v2)

SCREEN_PROMPT_BATCH = """Screen EACH numbered financial news article for its own target company.
Return JSON {"results": [{"id": <n>, "keep": true|false}, ...]} with exactly one entry per article.
Judge each article independently, using only its own text. This is a cheap first pass: when in doubt, keep=true.

""" + SCREEN_RULES                                     # ทีละ screen_batch ข่าว (screen-v3)


def screen_messages(row: dict, chars: int) -> list:
    msgs = build_messages(row, chars)                 # input เดียวกับรอบเต็ม แต่เนื้อข่าวสั้นกว่า
    return [{"role": "system", "content": SCREEN_PROMPT}, msgs[1]]           # system prompt สั้น ไม่ใช้ cache


def screen_one(row: dict, model: str, tries=3):
    err = None
    for _ in range(tries):
        text, usage = chat(screen_messages(row, C.screen_chars), model, max_tokens=50, response_format=SCREEN_FORMAT)
        try:
            m = re.search(r"\{.*\}", text.strip(), re.S)
            return ScreenLabel.model_validate(json.loads(m.group(0) if m else text)).keep, usage
        except (ValidationError, json.JSONDecodeError) as e:
            err = e
    raise RuntimeError(f"invalid screen JSON after {tries} tries: {str(err)[:200]}")


def _screen_item(i: int, row: dict, chars: int) -> str:
    body = f"{row.get('summary_src') or ''}\n{row.get('content') or ''}".strip()[:chars] or "(no body text)"
    return (f"[{i}] Target company: {row.get('company') or row['ticker']} (ticker {row['ticker']})\n"
            f"Headline: {row['title']}\nText: {' '.join(body.split())}")


def screen_batch_one(rows: list, model: str, tries=3):
    # คัดกรองหลายข่าวในการเรียกครั้งเดียว (system prompt ส่งครั้งเดียวต่อ batch) -> [keep, ...] ตามลำดับ rows
    msgs = [{"role": "system", "content": SCREEN_PROMPT_BATCH},
            {"role": "user", "content": "\n\n".join(_screen_item(i, r, C.screen_chars) for i, r in enumerate(rows, 1))}]
    err = None
    for _ in range(tries):
        text, usage = chat(msgs, model, max_tokens=40 * len(rows) + 50, response_format=SCREEN_BATCH_FORMAT)
        try:
            m = re.search(r"\{.*\}", text.strip(), re.S)
            res = {x.id: x.keep for x in ScreenBatch.model_validate(json.loads(m.group(0) if m else text)).results}
            if set(res) == set(range(1, len(rows) + 1)):
                return [res[i] for i in range(1, len(rows) + 1)], usage
            err = f"ids {sorted(res)[:5]}... != 1..{len(rows)}"
        except (ValidationError, json.JSONDecodeError) as e:
            err = e
    raise RuntimeError(f"invalid batch screen JSON after {tries} tries: {str(err)[:200]}")


def load_screen(model: str) -> dict:
    p = cache_path(model, C.screen_version)
    if not p.exists():
        return {}
    return {r["row_id"]: r for r in map(json.loads, filter(str.strip, p.read_text(encoding="utf-8").splitlines()))}


def screen_rows(df: pd.DataFrame, model: str, limit=None, workers=None) -> dict:
    # รอบคัดกรอง: ถามแค่ keep (cache แยกตาม screen_version) -> คืน {row_id: keep} ของทุกแถวใน df ที่คัดกรองแล้ว
    # screen_batch > 1: ส่งทีละ batch (ค่าใช้จ่ายของ batch หารเท่ากันทุกแถว) · batch ที่ fail = ยังไม่คัดกรอง (ส่งรอบเต็ม / รันซ้ำได้)
    done = load_screen(model)
    todo = [r for r in df.to_dict("records") if r["row_id"] not in done]
    if limit is not None:
        todo = todo[:limit]
    n = max(1, int(C.screen_batch))
    batches = [todo[i:i + n] for i in range(0, len(todo), n)]
    log(f"[screen] {model} | {C.screen_version} | batch {n} | cached {len(done)} | todo {len(todo)} ({len(batches)} calls)")
    cost, fails, lock, t0 = 0.0, 0, threading.Lock(), time.time()
    def run(b):
        if n > 1:
            return screen_batch_one(b, model)
        keep, usage = screen_one(b[0], model)
        return [keep], usage
    with open(cache_path(model, C.screen_version), "a", encoding="utf-8") as f, ThreadPoolExecutor(workers or C.llm_workers) as ex:
        futs = {ex.submit(run, b): b for b in batches}
        for i, fut in enumerate(as_completed(futs), 1):
            b = futs[fut]
            try:
                keeps, usage = fut.result()
            except Exception as e:          # noqa: BLE001
                fails += len(b)
                log(f"  ! batch {b[0]['row_id']} (+{len(b) - 1}): {str(e)[:160]}")
                continue
            c = float(usage.get("cost") or 0) / len(b)
            with lock:
                for r, keep in zip(b, keeps):
                    rec = {"row_id": r["row_id"], "model": model, "prompt_version": C.screen_version, "cost": c, "keep": bool(keep)}
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    done[r["row_id"]] = rec
                    cost += c
                f.flush()
            if i % 500 == 0:
                log(f"  {i}/{len(batches)} calls cost=${cost:.3f} ({time.time() - t0:.0f}s)")
    if todo:
        log(f"[screen] เสร็จ {len(todo) - fails}/{len(todo)} | fail {fails} | cost ${cost:.4f} "
            f"(${cost / max(1, len(todo) - fails):.6f}/แถว) | {time.time() - t0:.0f}s")
    return {k: done[k]["keep"] for k in df["row_id"] if k in done}


def screen_split(df: pd.DataFrame, model: str):
    # แบ่ง df -> (ส่งรอบเต็ม, ถูกคัดออก) · แถวที่มีป้าย v4 ใน cache แล้วใช้ป้ายนั้นเสมอ · ยังไม่คัดกรอง/ fail = ส่งรอบเต็ม
    if not C.screen:
        return df, df.iloc[0:0]
    full = load_cache(model)
    sc = load_screen(C.screen_model)
    out = df["row_id"].map(lambda k: k not in full and k in sc and not sc[k]["keep"])
    return df[~out], df[out]


def screen_labels(df: pd.DataFrame) -> pd.DataFrame:
    # ป้ายของแถวที่รอบคัดกรองตัด (ไม่ส่งรอบเต็ม) = relevant=false -> promotional ตามกฎข้อ 1 ของ v4
    return pd.DataFrame({
        "row_id": df["row_id"].to_numpy(), "model": "screen", "prompt_version": C.screen_version,
        "cost": 0.0, "reasoning": "screen:keep=false", "relevant": False, "important": False,
        "sentiment": 0, "summary": df["title"].to_numpy(), "primary_tag": "promotional",
        "secondary_tag": "none", "scheduled": False, "quantified": False, "rumor": False,
    })


def screen_cost_estimate(df: pd.DataFrame, model: str) -> dict:
    # ประมาณค่าใช้จ่ายที่เหลือของ df (= NEWS_LLM): คัดกรองแถวที่ยังไม่มีป้าย + รอบเต็มเฉพาะแถวที่ผ่าน
    full, sc = load_cache(model), load_screen(C.screen_model)
    sc_cost = [r["cost"] for r in sc.values() if r.get("cost")]
    s_cpr, f_cpr = (float(np.mean(sc_cost)) if sc_cost else float("nan")), cost_per_row(model)
    pass_rate = float(np.mean([r["keep"] for r in sc.values()])) if sc else float("nan")
    no_full = ~df["row_id"].isin(full)
    if not C.screen:
        n_full = int(no_full.sum())
        return {"screen_todo": 0, "full_todo": n_full, "cost_full": f_cpr * n_full, "total": f_cpr * n_full}
    s_todo = int((no_full & ~df["row_id"].isin(sc)).sum())
    kept_done = int((no_full & df["row_id"].map(lambda k: k in sc and sc[k]["keep"])).sum())
    n_full = kept_done + pass_rate * s_todo
    e = {"screen_cpr": s_cpr, "full_cpr": f_cpr, "pass_rate": pass_rate, "screen_todo": s_todo,
         "full_todo": round(n_full), "cost_screen": s_cpr * s_todo, "cost_full": f_cpr * n_full}
    e["total"] = e["cost_screen"] + e["cost_full"]
    return e


# ---- ตรวจรอบคัดกรอง (ต้องไม่ทิ้งข่าวที่ควรเก็บ) ----
_t = lambda s: s.astype(str).str.lower().isin(["1", "1.0", "true", "yes", "y"])     # noqa: E731
_keep_full = lambda d: d["relevant"].astype(bool) & d["important"].astype(bool)    # noqa: E731

# (ก) hand label 150 ข่าว (~$0.02): ข่าวที่คนเก็บต้องผ่านทุกข่าว
_h = pd.read_csv(P("hand_labels.csv"))
_hn = NEWS_LLM[NEWS_LLM["row_id"].isin(_h["row_id"])]
_sk = screen_rows(_hn, C.screen_model)
_h = _h[_h["row_id"].isin(_sk)].assign(screen_keep=lambda d: d["row_id"].map(_sk))
_hk = _t(_h["hand_relevant"]) & _t(_h["hand_important"])
print(f"(ก) hand label: คัดกรอง {len(_h)} ข่าว | ผ่าน {_h['screen_keep'].mean():.0%} | "
      f"ข่าวที่คนเก็บแต่ถูกตัด {int((_hk & ~_h['screen_keep']).sum())}/{int(_hk.sum())} (ต้องเป็น 0)")

# (ข) แถวที่มีป้าย v4 แล้ว (hand label + รันทดสอบ ม.ค. 2023, ~1,800 ข่าว, ~$0.3): ข่าวที่ v4 เก็บต้องผ่าน
_v4 = pd.DataFrame(load_cache(C.model).values())
if C.screen_validate_v4 and len(_v4):
    _vn = NEWS_LLM[NEWS_LLM["row_id"].isin(_v4["row_id"])]
    _sk = screen_rows(_vn, C.screen_model)
    _v = _v4[_v4["row_id"].isin(_sk)].assign(screen_keep=lambda d: d["row_id"].map(_sk))
    _vk = _keep_full(_v)
    print(f"(ข) เทียบป้าย v4: คัดกรอง {len(_v):,} ข่าว | ผ่าน {_v['screen_keep'].mean():.0%} | "
          f"ข่าวที่ v4 เก็บแต่ถูกตัด {int((_vk & ~_v['screen_keep']).sum())}/{int(_vk.sum())} "
          f"({(_vk & ~_v['screen_keep']).sum() / max(1, _vk.sum()):.1%}) | ส่งรอบเต็มเกินจำเป็น {int((~_vk & _v['screen_keep']).sum())}")
    _miss = _v[_vk & ~_v["screen_keep"]].merge(news[["row_id", "ticker", "title"]], on="row_id")
    if len(_miss):
        display(_miss[["ticker", "title", "primary_tag", "sentiment"]].head(30))

_e = screen_cost_estimate(NEWS_LLM, C.model)
print(f"ประมาณค่าใช้จ่ายที่เหลือ: คัดกรอง {_e['screen_todo']:,} × ${_e.get('screen_cpr', 0):.6f} = ${_e.get('cost_screen', 0):.2f} "
      f"+ รอบเต็ม ~{_e['full_todo']:,} × ${_e.get('full_cpr', float('nan')):.6f} = ${_e['cost_full']:.2f} "
      f"-> รวม ~${_e['total']:.2f} (งบ ${C.llm_budget_usd})")

### 2.3 รันชุดเต็ม → `news_tagged.parquet`
หยุดอัตโนมัติถ้าประมาณค่าใช้จ่ายเกิน `llm_budget_usd` · โหมด `test` ติดป้ายแค่ `llm_limit` แถว

In [ ]:
# งบรวม = รอบคัดกรอง + รอบเต็มเฉพาะแถวที่ผ่าน (ประมาณจากอัตราผ่านใน 2.2b) · ตรวจซ้ำก่อนเริ่มรอบเต็ม
_e = screen_cost_estimate(NEWS_LLM, C.model)
assert not (C.screen and not np.isfinite(_e.get("pass_rate", np.nan))), "รัน 2.2b ก่อน เพื่อให้มีอัตราผ่านไว้ประมาณค่าใช้จ่าย"
est = _e["total"] if C.llm_limit is None else np.nansum([_e.get("screen_cpr", 0), _e.get("full_cpr", 0)]) * C.llm_limit
print(f"ประมาณค่าใช้จ่าย ${est:.2f} (งบ ${C.llm_budget_usd}) | {_e}")
assert not (est > C.llm_budget_usd), f"ประมาณ ${est:.2f} เกินงบ ${C.llm_budget_usd} — ลดจำนวนบทความ / เพิ่มงบใน config"

if C.screen:
    _sc_before = sum(r["cost"] for r in load_screen(C.screen_model).values())
    screen_rows(NEWS_LLM[~NEWS_LLM["row_id"].isin(load_cache(C.model))], C.screen_model, limit=C.llm_limit)
    _sc_spent = sum(r["cost"] for r in load_screen(C.screen_model).values()) - _sc_before
else:
    _sc_spent = 0.0
FULL_ROWS, SCREENED_OUT = screen_split(NEWS_LLM, C.model)
_n_full = (~FULL_ROWS["row_id"].isin(load_cache(C.model))).sum()
_est_full = cost_per_row(C.model) * (_n_full if C.llm_limit is None else min(_n_full, C.llm_limit))
print(f"คัดกรองจ่าย ${_sc_spent:.2f} | ตัดออก {len(SCREENED_OUT):,} | รอบเต็ม {_n_full:,} แถว ~${_est_full:.2f}")
assert not (_sc_spent + _est_full > C.llm_budget_usd), f"รวม ${_sc_spent + _est_full:.2f} เกินงบ ${C.llm_budget_usd}"

labs = label_rows(FULL_ROWS, C.model, limit=C.llm_limit)
labs = pd.concat([labs, title_filter_labels(news, TF_RULE),        # ตัวกรองหัวข่าวตัด = ขยะ (ไม่เสียเงิน)
                  screen_labels(SCREENED_OUT)], ignore_index=True)  # รอบคัดกรองตัด = relevant=false -> promotional

tagged = news.merge(labs.drop(columns=["cost"], errors="ignore").rename(columns={"model": "label_source"}), on="row_id", how="inner")
tagged["is_junk"] = tagged["primary_tag"].isin(JUNK_TAGS)
tagged.to_parquet(P("news_tagged.parquet"), index=False)
keep = tagged["relevant"] & tagged["important"]
print(f"{len(tagged):,}/{len(news):,} แถวมีป้าย | relevant & important {keep.sum():,} ({keep.mean():.0%})")

### 2.4 นับข่าวต่อป้าย
ป้ายที่มีข่าวน้อยเกินไป → ใส่ใน `TAG_MERGE` เพื่อรวมกับป้ายอื่น (ใช้ตั้งแต่ขั้นตอนที่ 3)

In [ ]:
TAG_MERGE = {}          # เช่น {"dividend_buyback": "macro_other"}
k = tagged[tagged["relevant"] & tagged["important"]]
cnt = pd.crosstab(k["primary_tag"].map(lambda t: TAGS[t]), np.sign(k["sentiment"]).map({-1: "ลบ", 0: "กลาง", 1: "บวก"}), margins=True)
display(cnt)
display(tagged[["rumor", "quantified", "scheduled"]].mean().to_frame("สัดส่วน").T)

---
# ขั้นตอนที่ 3: จัดกลุ่มเป็นเรื่องและสร้างเหตุการณ์ (วันที่ 3)

- **story** = หลายบทความที่รายงานเหตุการณ์เดียวกัน (อาจหลายวัน) → ใช้ติดธง NEW / follow-up
- **event** = หน่วยที่วัดราคา = (หุ้น, วันซื้อขาย, ป้ายหลัก)

1. embedding `all-MiniLM-L6-v2` ของ หัวข่าว + summary (บันทึกเวกเตอร์ไว้)
2. **ปรับ threshold**: อ่าน `calibration_pairs.csv` (คู่ข่าว sim 0.70–0.90) หาจุดที่เริ่มเป็นคนละเรื่อง แล้วแก้ `sim_threshold`
3. greedy ภายใน (หุ้น, วัน, ป้าย) → เชื่อมข้ามวันกับ story ที่ยังเปิดของ (หุ้น, ป้าย) โดยเทียบ **centroid วันแรกที่ตรึงไว้**
4. อายุ story: M&A/คดีความ 75 วัน, ผู้บริหาร 15, ผลประกอบการ/นักวิเคราะห์ 5, อื่น ๆ 2
5. event: `n_articles, n_new, mean_sent, all_neutral, rumor, quantified, scheduled, sign` · เฉลี่ย 0 แต่ไม่เป็นกลางทุกบทความ = `mixed` (ตัดจากกลุ่มมีทิศทาง)

In [ ]:
def embed_cached(df: pd.DataFrame) -> np.ndarray:
    from sentence_transformers import SentenceTransformer
    vec_path, id_path = P("embeddings_minilm.npy"), P("embeddings_minilm_ids.parquet")
    if vec_path.exists() and id_path.exists():
        ids = pd.read_parquet(id_path)["row_id"]
        pos = pd.Series(np.arange(len(ids)), index=ids.values)
        if df["row_id"].isin(pos.index).all():
            return np.load(vec_path)[pos[df["row_id"]].to_numpy()]
    texts = (df["title"].fillna("") + ". " + df["summary"].fillna("")).tolist()
    vec = SentenceTransformer(C.embed_model).encode(texts, batch_size=64, show_progress_bar=len(texts) > 2000,
                                                    normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    np.save(vec_path, vec)
    df[["row_id"]].to_parquet(id_path, index=False)
    return vec


def _unit(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v


def build_stories(df: pd.DataFrame, E: np.ndarray, threshold: float) -> pd.DataFrame:
    df = df.reset_index(drop=True)
    story_of = np.empty(len(df), dtype=object)
    start_of = {}
    for (ticker, tag), g in df.groupby(["ticker", "primary_tag"], sort=False):
        max_age = STORY_MAX_AGE.get(tag, STORY_DEFAULT_AGE)
        open_stories, k = [], 0
        for td, gd in g.sort_values("published_utc").groupby("trade_date", sort=True):
            clusters = []                       # greedy ภายในวัน
            for i in gd.index:
                v = E[i]
                best, best_s = None, threshold
                for c in clusters:
                    s = float(v @ _unit(c["sum"]))
                    if s >= best_s:
                        best, best_s = c, s
                if best is None:
                    best = {"members": [], "sum": np.zeros_like(v)}
                    clusters.append(best)
                best["members"].append(i)
                best["sum"] = best["sum"] + v
            open_stories = [s for s in open_stories if (td - s["start"]).days <= max_age]
            for c in clusters:                  # เชื่อมข้ามวัน (centroid วันแรกตรึงไว้)
                cen = _unit(c["sum"])
                best, best_s = None, threshold
                for s in open_stories:
                    if s["start"] == td:
                        continue
                    sim = float(cen @ s["centroid"])
                    if sim >= best_s:
                        best, best_s = s, sim
                if best is None:
                    best = {"id": f"{ticker}|{tag}|{td:%Y%m%d}|{k}", "centroid": cen, "start": td}
                    k += 1
                    open_stories.append(best)
                    start_of[best["id"]] = td
                for i in c["members"]:
                    story_of[i] = best["id"]
    df["story_id"] = story_of
    df["story_start"] = df["story_id"].map(start_of)
    df["is_new"] = df["trade_date"] == df["story_start"]
    df["story_age"] = (df["trade_date"] - df["story_start"]).dt.days
    return df


def calibration_pairs(df, E, lo=0.70, hi=0.90, n=50, seed=0) -> pd.DataFrame:
    rows = []
    df = df.reset_index(drop=True)
    for _, g in df.groupby("ticker"):
        idx = g.index.to_numpy()
        if len(idx) < 2:
            continue
        if len(idx) > 1500:
            idx = np.random.default_rng(seed).choice(idx, 1500, replace=False)
        S = E[idx] @ E[idx].T
        a, b = np.triu_indices(len(idx), 1)
        s = S[a, b]
        m = (s >= lo) & (s <= hi)
        for i, j, v in zip(idx[a[m]], idx[b[m]], s[m]):
            rows.append({"sim": round(float(v), 3), "ticker": df.at[i, "ticker"], "title_a": df.at[i, "title"],
                         "title_b": df.at[j, "title"], "tag_a": df.at[i, "primary_tag"], "tag_b": df.at[j, "primary_tag"]})
    out = pd.DataFrame(rows)
    if len(out) > n:
        out = out.sample(n, random_state=seed)
    out["same_story(fill)"] = ""
    return out.sort_values("sim").reset_index(drop=True)


def build_events(st: pd.DataFrame) -> pd.DataFrame:
    g = st.groupby(["ticker", "trade_date", "primary_tag"], sort=True)
    ev = g.agg(n_articles=("row_id", "size"), n_new=("is_new", "sum"), mean_sent=("sentiment", "mean"),
               all_neutral=("sentiment", lambda s: bool((s == 0).all())), rumor=("rumor", "any"),
               quantified=("quantified", "any"), scheduled=("scheduled", "any"),
               n_stories=("story_id", "nunique"), headline=("title", "first")).reset_index()
    ev["sign"] = np.sign(ev["mean_sent"]).astype(int)
    ev["direction"] = np.where(ev["all_neutral"], "neutral",
                               np.where(ev["sign"] > 0, "positive", np.where(ev["sign"] < 0, "negative", "mixed")))
    ev["is_junk"] = ev["primary_tag"].isin(JUNK_TAGS)
    nz = ev[~ev["is_junk"]].groupby(["ticker", "trade_date"])["primary_tag"].nunique().rename("n_tags_day")
    ev = ev.merge(nz, on=["ticker", "trade_date"], how="left")
    ev["n_tags_day"] = ev["n_tags_day"].fillna(0).astype(int)
    ev.insert(0, "event_id", [f"ev{i:06d}" for i in range(len(ev))])
    return ev


tagged = pd.read_parquet(P("news_tagged.parquet"))
tagged["primary_tag"] = tagged["primary_tag"].replace(TAG_MERGE)
df3 = tagged[tagged["relevant"] & tagged["important"] & tagged["trade_date"].notna()]
df3 = df3.sort_values(["ticker", "published_utc"]).reset_index(drop=True)
E = embed_cached(df3)
pairs = calibration_pairs(df3, E)
pairs.to_csv(P("calibration_pairs.csv"), index=False, encoding="utf-8-sig")
print(f"{len(df3):,} แถว relevant & important | คู่ทดสอบ threshold {len(pairs)} คู่ -> calibration_pairs.csv")
display(pairs[["sim", "ticker", "title_a", "title_b"]].head(20))

In [ ]:
stories = build_stories(df3, E, C.sim_threshold)
stories.to_parquet(P("news_stories.parquet"), index=False)
events = build_events(stories)
events.to_parquet(P("events.parquet"), index=False)
real = events[~events["is_junk"]]
print(f"{stories['story_id'].nunique():,} stories | follow-up {1 - stories['is_new'].mean():.0%} (เปเปอร์ ~55%)")
print(f"{len(events):,} เหตุการณ์ (ไม่ใช่ขยะ {len(real):,}) | ข่าวเป็นกลาง {(real['direction'] == 'neutral').mean():.0%} (เปเปอร์ ~22%)")
display(real.groupby(["primary_tag", "direction"]).size().unstack(fill_value=0))

In [ ]:
# ตรวจด้วยตา: สุ่มอ่าน 20 story ที่มีหลายบทความ ว่าเป็นเรื่องเดียวกันจริงไหม
multi = stories[stories.duplicated("story_id", keep=False)]
for sid in pd.Series(multi["story_id"].unique()).sample(min(20, multi["story_id"].nunique()), random_state=0):
    g = multi[multi["story_id"] == sid]
    print(f"## {sid}  ({len(g)} บทความ)")
    for _, r in g.head(6).iterrows():
        print(f"   {'NEW' if r['is_new'] else 'fu '} {r['trade_date']:%Y-%m-%d} {r['title'][:110]}")

---
# ขั้นตอนที่ 4: วัดผลตอบแทนผิดปกติ (วันที่ 4)

$AR_t = r_t - \beta_t \cdot r_{SPY,t}$ — β = cov/var rolling 252 วัน (min 126) `shift(1)` ใช้ข้อมูลถึงเมื่อวานเท่านั้น, ไม่มีค่า → 1

- CAR 4 ช่วง: pre [−5, 0], d0 [0], post1 [+1, +5], post2 [+6, +20] · ข้ามเหตุการณ์ที่ราคาไม่ครบ −5..+20
- **เก็บ CAR ดิบเสมอ** คูณ `sign` เฉพาะตอนสรุป · เก็บ AR รายวัน −5..+20 สำหรับกราฟ
- กลุ่มขนาด: มูลค่าซื้อขายเฉลี่ยรายปี → `qcut(3)` แยกทีละปี (ในหุ้น S&P 500 = ใหญ่ / ใหญ่กว่า / ใหญ่ที่สุด)
- p-value: bootstrap สุ่มวันซื้อขายทั้งวัน

In [ ]:
def load_wide():
    px = pd.read_parquet(P("prices.parquet"))
    return (px.pivot(index="date", columns="ticker", values="close").sort_index(),
            px.pivot(index="date", columns="ticker", values="volume").sort_index())


def compute_ar(close, factors=("SPY",), window=None, min_periods=None):
    window, min_periods = window or C.beta_window, min_periods or C.beta_min_periods
    r = close.pct_change(fill_method=None)
    stocks = [c for c in r.columns if c not in BENCHMARKS]
    factors = list(factors)
    if len(factors) == 1:
        m = r[factors[0]]
        beta = (r[stocks].rolling(window, min_periods=min_periods).cov(m)
                .div(m.rolling(window, min_periods=min_periods).var(), axis=0).shift(1).fillna(1.0))
        return r[stocks] - beta.mul(m, axis=0), beta
    # หลายปัจจัย (ขั้นตอนที่ 6): rolling OLS มีค่าคงที่ จาก rolling sum ของ X'X และ X'y
    F = r[factors]
    ar = pd.DataFrame(index=r.index, columns=stocks, dtype=float)
    beta = pd.DataFrame(index=r.index, columns=stocks, dtype=float)
    p = len(factors) + 1
    for s in stocks:
        y = r[s]
        ok = y.notna() & F.notna().all(axis=1)
        X = pd.concat([pd.Series(1.0, index=r.index, name="const"), F], axis=1).where(ok, 0.0)
        yy = y.where(ok, 0.0)
        roll = lambda z: z.rolling(window, min_periods=1).sum().to_numpy()       # noqa: E731
        nobs = ok.astype(float).rolling(window, min_periods=1).sum().to_numpy()
        XtX, Xty, cols = np.empty((len(r), p, p)), np.empty((len(r), p)), list(X.columns)
        for i in range(p):
            Xty[:, i] = roll(X[cols[i]] * yy)
            for j in range(i, p):
                XtX[:, i, j] = XtX[:, j, i] = roll(X[cols[i]] * X[cols[j]])
        coef = np.full((len(r), p), np.nan)
        good = nobs >= min_periods
        if good.any():
            coef[good] = np.array([np.linalg.solve(a, v) if np.linalg.cond(a) < 1e12 else np.full(p, np.nan)
                                   for a, v in zip(XtX[good], Xty[good])])
        b = pd.DataFrame(coef[:, 1:], index=r.index, columns=factors).shift(1)
        b[factors[0]] = b[factors[0]].fillna(1.0)
        b = b.fillna(0.0)
        ar[s] = y - (b * F).sum(axis=1, min_count=1)
        beta[s] = b[factors[0]]
    return ar, beta


def size_buckets(close, vol) -> pd.DataFrame:
    stocks = [c for c in close.columns if c not in BENCHMARKS]
    dv = (close[stocks] * vol[stocks]).groupby(close.index.year).mean()
    rows = []
    for y, s in dv.iterrows():
        s = s.dropna()
        if len(s) >= 3:
            q = pd.qcut(s.rank(method="first"), 3, labels=[0, 1, 2]).astype(int)
            rows += [{"year": y, "ticker": t, "size_bucket": b} for t, b in q.items()]
    return pd.DataFrame(rows)


def attach_size(ev, sb):
    ev = ev.copy()
    ev["year"] = ev["trade_date"].dt.year
    years = sorted(sb["year"].unique())
    ev["_y"] = ev["year"].map(lambda y: max([v for v in years if v <= y], default=years[0]))
    return ev.merge(sb.rename(columns={"year": "_y"}), on=["_y", "ticker"], how="left").drop(columns="_y")


def add_returns(ev, ar, beta, cal=CAL):
    ev = ev.reset_index(drop=True).copy()
    A = ar.reindex(cal)
    pos = cal.get_indexer(ev["trade_date"])
    col = A.columns.get_indexer(ev["ticker"])
    off = np.array(OFFSETS)
    rows = pos[:, None] + off[None, :]
    ok = (pos[:, None] >= 0) & (col[:, None] >= 0) & (rows >= 0) & (rows < len(A))
    vals = np.full(rows.shape, np.nan)
    vals[ok] = A.to_numpy(float)[rows[ok], np.broadcast_to(col[:, None], rows.shape)[ok]]
    ev[AR_COLS] = vals
    for w, (a, b) in WINDOWS.items():
        sub = vals[:, (off >= a) & (off <= b)]
        ev[CAR_COLS[w]] = np.where(np.isnan(sub).any(1), np.nan, np.nansum(sub, 1))
    ev["complete"] = ~np.isnan(vals).any(1)
    B = beta.reindex(cal)
    okb = (pos >= 0) & (col >= 0)
    ev["beta"] = np.nan
    ev.loc[okb, "beta"] = B.to_numpy(float)[pos[okb], col[okb]]
    return ev


def directional(ev):
    return ev[(~ev["is_junk"]) & ev["direction"].isin(["positive", "negative"])]


def cum_curve(df, by=None, baseline=None) -> pd.DataFrame:
    # ค่าเฉลี่ย AR x sign (หักค่าฐานรายวันถ้ามี) สะสมจากวัน -5
    curves = {}
    for g, d in ([("ALL", df)] if by is None else df.groupby(by)):
        A = d[AR_COLS].to_numpy(float)
        if baseline is not None:
            A = A - np.vstack([baseline.get(int(b) if pd.notna(b) else 1, np.zeros(len(AR_COLS))) for b in d["size_bucket"]])
        A = A * d["sign"].to_numpy()[:, None]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            mu = np.nanmean(A, 0) if len(A) else np.full(len(AR_COLS), np.nan)
        cum = np.nancumsum(mu) * 100
        cum[np.isnan(mu)] = np.nan
        curves[g] = cum
    return pd.DataFrame(curves, index=OFFSETS)


def plot_curves(curves, title, fname):
    fig, ax = plt.subplots(figsize=(9, 4.8))
    for c in curves.columns:
        ax.plot(curves.index, curves[c], marker="o", ms=2.5, lw=1.4, label=str(c))
    ax.axvline(0, color="grey", lw=0.8, ls="--")
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_xlabel("trading day relative to news (0 = news day)")
    ax.set_ylabel("cumulative signed AR (%)")
    ax.set_title(title)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(P(fname), dpi=130)
    plt.show()


SCOLS = {w: f"s_{c}" for w, c in CAR_COLS.items()}


def signed(d):
    d = d.copy()
    for c in CAR_COLS.values():
        d[f"s_{c}"] = d[c] * d["sign"]
    return d

In [ ]:
events = pd.read_parquet(P("events.parquet"))
close, vol = load_wide()
ar, beta = compute_ar(close, ("SPY",))
ar.to_parquet(P("ar_daily.parquet"))
SB = size_buckets(close, vol)
ev4 = attach_size(add_returns(events, ar, beta), SB)
n0 = len(ev4)
ev4 = ev4[ev4["ticker"].isin(ar.columns)]
n_nopx, n_inc = n0 - len(ev4), int((~ev4["complete"]).sum())
ev4 = ev4[ev4["complete"]].reset_index(drop=True)
ev4.to_parquet(P("event_returns.parquet"), index=False)
print(f"{n0:,} เหตุการณ์ | ไม่มีราคา {n_nopx} | ราคาไม่ครบ -5..+20 {n_inc} -> {len(ev4):,} "
      f"(ข้าม {(n_nopx + n_inc) / max(1, n0):.1%}, เปเปอร์ 9.7%)")

d4 = signed(directional(ev4))
raw_tab = pd.concat([summarize(d4, SCOLS), summarize(d4, SCOLS, "primary_tag")], ignore_index=True)
raw_tab.to_csv(P("tables") / "step4_raw_by_tag.csv", index=False)
print("ค่าเฉลี่ย CAR x sign (%) ของเหตุการณ์มีทิศทาง — raw (ยังมี drift พื้นหลังปน)")
display(wide(raw_tab).round(3).join(wide(raw_tab, "n")["pre"].rename("n")))
print("p-value (bootstrap วันซื้อขาย):")
display(wide(raw_tab, "p").round(3))

In [ ]:
curves4 = cum_curve(d4).join(cum_curve(d4, "primary_tag"))
curves4.to_csv(P("tables") / "step4_raw_cum_curve.csv")
plot_curves(curves4, f"{MODE}: raw signed CAR (-5..+20)", "fig_step4_raw_cum.png")
c_all = curves4["ALL"]
ratio = c_all.loc[0] / c_all.loc[20] if np.isfinite(c_all.loc[20]) and c_all.loc[20] != 0 else np.nan
print(f"ผลสะสม ณ ปิดวันข่าว {c_all.loc[0]:.3f}% / วัน +20 {c_all.loc[20]:.3f}% -> ratio {ratio:.2f} (เปเปอร์ 2.8)")

# ค่าสุดโต่ง: ดูผลแบบ winsorize 1% ประกอบ
dw = signed(winsorize_cols(directional(ev4), list(CAR_COLS.values()), C.winsor))
print(f"winsorize {C.winsor:.0%}:")
display(wide(summarize(dw, SCOLS)).round(3))

---
# ขั้นตอนที่ 5: เทียบกับ placebo (วันที่ 5) — หัวใจของงาน

benchmark beta เดียวไม่พอดี → มี drift ตกค้างในทุกช่วง · **ข่าวเป็นกลาง** = การเป็นข่าวโดยไม่มีทิศทาง → AR รอบข่าวเหล่านี้ = ค่าฐาน

- ค่าฐาน $b(k, w)$ = CAR เฉลี่ยของข่าวเป็นกลาง แยก 3 กลุ่มขนาด × 4 ช่วง (12 ค่า) รวมทุกประเภทข่าว
- $\text{adjusted} = s \cdot (CAR_w - b_{k,w})$ — หักค่าฐานก่อน แล้วคูณเครื่องหมาย (ใช้ CAR ดิบเท่านั้น)
- ข่าวเป็นกลาง < `min_neutral_per_bucket` ต่อกลุ่ม → ใช้ **วันเงียบ** (หุ้น, วัน ที่ไม่มีเหตุการณ์ในวัน −5..0) แทน
- ห้ามรวมข่าวเป็นกลางในค่าเฉลี่ยกลุ่มมีทิศทาง · กลุ่มขนาดแบ่งวิธีเดียวกับขั้นตอนที่ 4

In [ ]:
def quiet_days(ev, tickers, n, seed=0) -> pd.DataFrame:
    days = CAL[(CAL >= pd.Timestamp(C.news_start)) & (CAL <= pd.Timestamp(C.news_end))]
    pos_of = {d: i for i, d in enumerate(CAL)}
    busy = set()
    real = ev[~ev["is_junk"]]
    for t, d in zip(real["ticker"], real["trade_date"]):
        p = pos_of.get(d)
        if p is not None:
            for k in range(6):             # เหตุการณ์วัน p ทำให้วัน p..p+5 ไม่เงียบ
                if p + k < len(CAL):
                    busy.add((t, CAL[p + k]))
    cand = [(t, d) for t in tickers for d in days if (t, d) not in busy]
    rng = np.random.default_rng(seed)
    if len(cand) > n:
        cand = [cand[i] for i in rng.choice(len(cand), n, replace=False)]
    return pd.DataFrame(cand, columns=["ticker", "trade_date"])


def _base_table(df, source):
    rows, daily = [], {}
    for b, g in df.groupby("size_bucket"):
        for w, c in CAR_COLS.items():
            rows.append({"source": source, "size_bucket": int(b), "window": w, "b": g[c].mean(), "n": int(g[c].notna().sum())})
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            daily[int(b)] = np.nan_to_num(np.nanmean(g[AR_COLS].to_numpy(float), 0))
    return pd.DataFrame(rows), daily


def make_baseline(ev_ret, ar, beta, sb, verbose=True):
    neutral = ev_ret[(~ev_ret["is_junk"]) & (ev_ret["direction"] == "neutral")]
    bn, dn = _base_table(neutral, "neutral")
    q = quiet_days(ev_ret, list(ar.columns), C.n_quiet, C.seed)
    bq, dq = pd.DataFrame(), {}
    if len(q):
        q = attach_size(add_returns(q.assign(is_junk=False, direction="quiet", sign=0), ar, beta), sb)
        bq, dq = _base_table(q[q["complete"]], "quiet")
    base = pd.concat([bn, bq], ignore_index=True)
    n_neu = neutral.groupby("size_bucket").size().reindex([0, 1, 2]).fillna(0)
    use = "neutral" if n_neu.min() >= C.min_neutral_per_bucket or bq.empty else "quiet"
    if verbose:
        log(f"[placebo] ข่าวเป็นกลาง/กลุ่มขนาด {n_neu.astype(int).tolist()} "
            f"({len(neutral) / max(1, (~ev_ret['is_junk']).sum()):.0%} ของเหตุการณ์, เปเปอร์ ~22%) "
            f"| วันเงียบ {len(q):,} -> ใช้ค่าฐาน '{use}'")
    chosen = base[base["source"] == use]
    bmap = {(r.size_bucket, r.window): r.b for r in chosen.itertuples()}
    overall = chosen.groupby("window")["b"].mean().to_dict()
    for k in (0, 1, 2):
        for w in WINDOWS:
            v = bmap.get((k, w), np.nan)
            bmap[(k, w)] = v if np.isfinite(v) else float(np.nan_to_num(overall.get(w, 0.0)))
    return base.assign(used=base["source"] == use), use, bmap, (dn if use == "neutral" else dq)


def apply_adjust(ev_ret, bmap):
    ev = ev_ret.copy()
    k = ev["size_bucket"].fillna(1).astype(int)
    for w, c in CAR_COLS.items():
        ev[f"b_{w}"] = np.array([bmap[(kk, w)] for kk in k])
        ev[ADJ_COLS[w]] = ev["sign"] * (ev[c] - ev[f"b_{w}"])
        ev[f"s_{c}"] = ev["sign"] * ev[c]
    is_dir = (~ev["is_junk"]) & ev["direction"].isin(["positive", "negative"])
    for c in list(ADJ_COLS.values()) + list(SCOLS.values()):
        ev.loc[~is_dir, c] = np.nan            # ข่าวเป็นกลางไม่เข้าค่าเฉลี่ยกลุ่มมีทิศทาง
    return ev


def raw_vs_adj(d, by=None):
    return pd.concat([summarize(d, SCOLS, by).assign(kind="raw"), summarize(d, ADJ_COLS, by).assign(kind="adjusted")],
                     ignore_index=True)


def side_by_side(t):
    return pd.concat({k: wide(t[t["kind"] == k]) for k in ["raw", "adjusted"]}, axis=1).round(3)


ev_ret = pd.read_parquet(P("event_returns.parquet"))
base, BASE_USED, BMAP, DAILY = make_baseline(ev_ret, ar, beta, SB)
base.to_parquet(P("baseline.parquet"), index=False)
ev5 = apply_adjust(ev_ret, BMAP)
ev5.to_parquet(P("event_returns_adj.parquet"), index=False)
print("ค่าฐาน (% CAR) — ข่าวเป็นกลาง vs วันเงียบ (เปเปอร์ d0..: เป็นกลาง −0.92/−0.58/−0.34, วันเงียบ −0.74/−0.88/−0.59 เล็ก/กลาง/ใหญ่):")
display(base.pivot_table(index=["source", "size_bucket"], columns="window", values="b")[list(WINDOWS)].mul(100).round(3))

In [ ]:
d5 = directional(ev5)
main5 = pd.concat([raw_vs_adj(d5), raw_vs_adj(d5, "primary_tag")], ignore_index=True)
main5.to_csv(P("tables") / "step5_main_raw_vs_adj.csv", index=False)
print("ตารางหลัก: raw vs adjusted (%)")
t5 = side_by_side(main5)
t5[("", "n")] = wide(main5[main5["kind"] == "adjusted"], "n")["pre"]
display(t5)
print("p-value adjusted:")
display(wide(main5[main5["kind"] == "adjusted"], "p").round(3))

In [ ]:
# ทดสอบภาพลวงตา: ข่าวดี/ข่าวร้าย และกลุ่มขนาด — ความต่างแบบ raw ควรหายไปหลัง adjusted
gb5 = raw_vs_adj(d5, "direction")
sz5 = raw_vs_adj(d5, "size_bucket")
gb5.to_csv(P("tables") / "step5_good_vs_bad.csv", index=False)
sz5.to_csv(P("tables") / "step5_by_size.csv", index=False)
print("ข่าวดี / ข่าวร้าย (เปเปอร์ post: raw −0.62% / +0.63% -> adjusted 0.00% ทั้งคู่)")
display(side_by_side(gb5))
print("กลุ่มขนาด 0=ใหญ่ 1=ใหญ่กว่า 2=ใหญ่ที่สุด (เปเปอร์: หุ้นเล็กกลับตัวมากกว่าหุ้นใหญ่ 2.4 เท่า ใน raw แล้วหายไป)")
display(side_by_side(sz5))

c5 = pd.DataFrame({"raw": cum_curve(d5)["ALL"], "adjusted": cum_curve(d5, baseline=DAILY)["ALL"]})
plot_curves(c5, f"{MODE}: raw vs placebo-adjusted (baseline = {BASE_USED})", "fig_step5_raw_vs_adj.png")
gbc = pd.concat({f"{k} {lab}": cum_curve(d5[d5["direction"] == k], baseline=bl)["ALL"]
                 for k in ["positive", "negative"] for lab, bl in [("raw", None), ("adj", DAILY)]}, axis=1)
plot_curves(gbc, f"{MODE}: good vs bad news", "fig_step5_good_vs_bad.png")
pd.concat([c5, gbc], axis=1).to_csv(P("tables") / "step5_cum_curves.csv")

---
# ขั้นตอนที่ 6: ตรวจความทนทาน (วันที่ 6)

**ควรทำ** (ทุก variant ใช้ค่า adjusted)
1. NEW only: `n_new > 0`
2. วันข่าวประเภทเดียว: (หุ้น, วัน) ที่มี 1 ป้าย
3. รายปี 2021 / 2022 / 2023 (ขาขึ้น / ขาลง / ฟื้นตัว)
4. Benjamini-Hochberg FDR 5% ภายในแต่ละ variant (`multipletests(..., "fdr_bh")`)
5. ป้ายที่ 2 โมเดลตรงกัน: ติดป้ายซ้ำ `second_model_n` บทความด้วย `second_model` แล้วสร้างเหตุการณ์จากบทความที่ป้ายหลักตรงกันเท่านั้น
6. bootstrap 5,000 รอบ (โหมด full)

**ทำถ้ามีเวลา**: คุณลักษณะ (`scheduled, quantified, rumor`), benchmark 3 ปัจจัย SPY + GLD + IEF (rolling OLS → ขั้นตอนที่ 4–5 ซ้ำ)

In [ ]:
DO_TWO_MODEL = True
DO_3FACTOR = True


def adj_table(d, variant):
    return pd.concat([summarize(d, ADJ_COLS), summarize(d, ADJ_COLS, "primary_tag")], ignore_index=True).assign(variant=variant)


def events_to_adj(ev, ar_, beta_, bmap=None):
    ev = attach_size(add_returns(ev, ar_, beta_), SB)
    ev = ev[ev["ticker"].isin(ar_.columns) & ev["complete"]]
    if bmap is None:
        _, _, bmap, _ = make_baseline(ev, ar_, beta_, SB, verbose=False)
    return apply_adjust(ev, bmap)


ev6 = pd.read_parquet(P("event_returns_adj.parquet"))
d6 = directional(ev6)
tabs = [adj_table(d6, "all"), adj_table(d6[d6["n_new"] > 0], "new_only"), adj_table(d6[d6["n_tags_day"] == 1], "single_tag_day")]
for y, g in d6.groupby(d6["trade_date"].dt.year):
    tabs.append(adj_table(g, f"year_{y}"))
INFO = {}

if DO_TWO_MODEL:
    st6 = pd.read_parquet(P("news_stories.parquet"))
    sample = st6.sample(min(C.second_model_n, len(st6)), random_state=C.seed)
    src = pd.read_parquet(P("news_tagged.parquet"))
    labs2 = label_rows(src[src["row_id"].isin(sample["row_id"])], C.second_model)
    if len(labs2):
        m2 = sample.merge(labs2[["row_id", "primary_tag", "sentiment"]].rename(
            columns={"primary_tag": "tag2", "sentiment": "sent2"}), on="row_id", how="inner")
        agree = m2["primary_tag"] == m2["tag2"].replace(TAG_MERGE)
        INFO = {"n_double_labeled": len(m2), "tag_agreement": round(float(agree.mean()), 3),
                "sentiment_sign_agreement": round(float((np.sign(m2["sentiment"]) == np.sign(m2["sent2"])).mean()), 3)}
        tabs.append(adj_table(directional(events_to_adj(build_events(m2[agree]), ar, beta, BMAP)), "two_model_agree"))

if DO_3FACTOR:
    ar3, beta3 = compute_ar(close, C.factors3)          # ค่าฐานประมาณใหม่ด้วย AR 3 ปัจจัย
    tabs.append(adj_table(directional(events_to_adj(pd.read_parquet(P("events.parquet")), ar3, beta3)),
                          "3factor_" + "+".join(C.factors3)))

rob = pd.concat(tabs, ignore_index=True)
rob["q"] = np.nan
for v, g in rob.groupby("variant"):                    # BH ภายในแต่ละ variant (ไม่รวมแถว ALL)
    idx = g.index[g["group"] != "ALL"]
    rob.loc[idx, "q"] = bh_qvalues(rob.loc[idx, "p"])
rob.to_csv(P("tables") / "step6_robustness.csv", index=False)
print("2 โมเดล:", INFO)
robust_wide = rob.pivot_table(index=["group", "window"], columns="variant", values="mean_pct")
robust_wide.to_csv(P("tables") / "step6_robustness_wide.csv")
print("ตารางความทนทาน (adjusted CAR %, ช่วง post1 และ post2):")
display(robust_wide.loc[(slice(None), ["post1", "post2"]), :].round(3))
print("จำนวนเหตุการณ์ต่อ variant:")
display(rob[rob["group"] == "ALL"].groupby("variant")["n"].max().to_frame().T)

In [ ]:
# q-value (BH) และตาราง drift ฉบับสุดท้าย: *** q<0.01, ** q<0.05, * q<0.10
print("q-value (variant = all):")
display(wide(rob[rob["variant"] == "all"], "q").round(3))
main6 = rob[rob["variant"] == "all"].copy()
star = np.select([main6["q"] < 0.01, main6["q"] < 0.05, main6["q"] < 0.10], ["***", "**", "*"], "")
main6["cell"] = main6["mean_pct"].map(lambda v: f"{v:+.2f}" if np.isfinite(v) else "") + star
drift = main6.pivot(index="group", columns="window", values="cell")[list(WINDOWS)]
drift["n"] = main6.groupby("group")["n"].max()
drift.index = [TAGS.get(i, i) for i in drift.index]
drift.to_csv(P("tables") / "step6_final_drift_table.csv")
print("ตาราง drift ฉบับสุดท้าย (adjusted, %):")
display(drift)

# คุณลักษณะ และขนาดหุ้น
attrs = pd.concat([summarize(d6, ADJ_COLS, a).assign(attribute=a) for a in ["scheduled", "quantified", "rumor"]], ignore_index=True)
attrs.to_csv(P("tables") / "step6_attributes.csv", index=False)
print("คุณลักษณะ (เปเปอร์: มีกำหนดการไหลต่อ +0.08%, ไม่มีตัวเลขกลับตัว −0.12%):")
display(pd.concat({a: wide(g) for a, g in attrs.groupby("attribute")}).round(3))
size6 = summarize(d6, ADJ_COLS, "size_bucket")
size6.to_csv(P("tables") / "step6_by_size.csv", index=False)
display(wide(size6).round(3))

### เทียบกับเปเปอร์
ตัวเลขของเปเปอร์ = adjusted drift วัน +6..+20 (ตาราง 2) จึงเทียบกับ `post2` ของเรา · เปเปอร์ใช้ 17 ป้าย — ป้ายที่เรารวม (ผลประกอบการ + guidance, เปิดตัวสินค้า + พันธมิตร) ใช้ค่าเฉลี่ยคร่าว ๆ ของเปเปอร์เป็นตัวอ้างอิงเท่านั้น

In [ ]:
PAPER = {  # adjusted drift หลังข่าวจากเปเปอร์ (%)
    "dividend_buyback": 0.35, "earnings_guidance": (0.22 + 0.13) / 2, "analyst": 0.10,
    "macro_other": -0.34, "product_partnership": -0.18, "management": -0.18, "legal_regulatory": -0.08,
}
ours = main6.pivot(index="group", columns="window", values="mean_pct")
# เปเปอร์ตาราง 2 "adj. +6..+20" = ช่วง post2 เท่านั้น -> เทียบ post2 ต่อ post2 (post1 แสดงไว้ดูประกอบ)
cmp_ = pd.DataFrame({"paper_post2": pd.Series(PAPER), "ours_post2": ours["post2"], "ours_post1": ours["post1"]}).round(3)
cmp_["same_sign"] = (np.sign(cmp_["paper_post2"]) == np.sign(cmp_["ours_post2"])).where(cmp_["paper_post2"].notna())
cmp_.index = [TAGS.get(i, i) for i in cmp_.index]
display(cmp_)
summary = {
    "mode": MODE, "universe": len(universe) if "universe" in dir() else None,
    "articles_tagged": len(pd.read_parquet(P("news_tagged.parquet"))),
    "events_measured": len(ev6), "directional": len(d6),
    "follow_up_share": round(1 - pd.read_parquet(P("news_stories.parquet"))["is_new"].mean(), 3),
    "neutral_share": round(float((ev6.loc[~ev6["is_junk"], "direction"] == "neutral").mean()), 3),
    "ratio_d0_to_d20_raw": round(float(ratio), 2) if np.isfinite(ratio) else None,
    "baseline_used": BASE_USED, **INFO,
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
summary

---
## ข้อจำกัด (เขียนในรายงาน)
- **Look-ahead**: LLM ติดป้ายย้อนหลังและอาจรู้ผลในอนาคต (prompt สั่งให้ตัดสินจากเนื้อข่าวเท่านั้น) ต่างจากเปเปอร์ที่สร้าง sentiment วันต่อวัน
- **Survivorship bias**: ใช้รายชื่อ S&P 500 ปัจจุบัน + ราคา yfinance → ไม่มีบริษัทที่หลุดจากดัชนี
- **หุ้นใหญ่เท่านั้น**: drift และกลยุทธ์ในเปเปอร์ชัดที่สุดในหุ้นเล็ก ซึ่งไม่อยู่ในงานนี้
- **ขนาดข้อมูล**: เล็กกว่าเปเปอร์หลายสิบเท่า และใช้ป้ายน้อยกว่า (8 + ขยะ 2 vs 17)
- **เวลาใน FNSPID**: ข่าวส่วนใหญ่มีแค่วันที่ (00:00) → ใช้กฎ 16:00 ไม่ได้ ข่าวหลังปิดตลาดอาจถูกนับเป็นวันก่อนหน้า 1 วัน (ช่วง pre รวมวัน 0 จึงยังครอบคลุม)
- **FNSPID ไม่ครบ**: ถ้าไฟล์ `nasdaq_exteral_data.csv` ยังดาวน์โหลดไม่ครบ หุ้นท้ายตัวอักษร (หลัง `last_symbol` ใน `fnspid_sp500_2021_2023.json`) ถูกตัดออกจาก universe

## งานขยายรอบถัดไป
- เพิ่มหุ้นเล็กและกลางจาก Nasdaq หรือ S&P 600 ราว 100 ตัว · benchmark 3–4 ปัจจัย (GLD, IEF, IWM)
- วิเคราะห์ข่าวลือแบบเต็ม, ความผันผวน และพอร์ตจำลองที่มี "short หุ้นที่เป็นข่าวทุกตัว" เป็นตัวเทียบ